In [ ]:
# ======================================================================================
# CCD_READABLE_SINGLE_CELL_V1
# UNIFORM CCD v1 — SINGLE-CELL KAGGLE NOTEBOOK
# Scientific protocol: supplied ccd_uniform_pipeline_v1 ZIP.
# Presentation layout: supplied LLAMA-3.1-8B-INSTRUCT notebook.
# ======================================================================================
# Enable GPU + Internet and run this cell in a fresh Kaggle session.
# No attached dataset/config files are required. Only the selected dataset is downloaded.
# This is ordinary Python: no serialized source-code payload and no hidden worker script.
#
# The completed run writes a *_FROZEN.ipynb notebook containing the exact sampled records,
# model commits and runtime package versions. Use that notebook for repeat runs and the
# other models on this dataset; change only MODEL_KEY. It needs no input attachments.
# Fresh runs of THIS unfrozen notebook bootstrap an environment; they are not a substitute
# for replaying the frozen notebook. Version pins do not guarantee bitwise GPU determinism.
# Preserve the runtime image, model files and package artifacts for long-term reproduction.
#
# Original coordinator/assignment/validation/aggregation functions remain available below.
# Final aggregation uses ACTION="aggregate" after all 60 raw run folders are collected.
from __future__ import annotations

# -----------------------------
# 0) Job selection — edit only the assigned model and dataset
# -----------------------------
MODEL_KEY = "gemma3_270m"
DATASET_NAME = "gsm8k"                 # gsm8k | strategyqa | logiqa | mbpp
ALLOW_CODE_EXECUTION = False          # Explicitly enable only for MBPP, as in the ZIP README.
ACTION = "run"                       # run | aggregate
WORK_ROOT = "/kaggle/working/ccd_uniform"
OUTPUT_ROOT = "/kaggle/working"
AGGREGATE_RESULTS_ROOT = "/kaggle/working/ccd_uniform/results"

# Populated automatically in the exported *_FROZEN.ipynb. Do not edit its contents.
REPLAY_STATE = None

# -----------------------------
# 1) Minimal environment setup — before importing Unsloth/Transformers
# -----------------------------
import os
# Compatibility/reproducibility guard: the protocol defines an effective batch size of 16
# from per-device batch 1 and gradient accumulation 16, so expose exactly one GPU.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
import re
import json
import ast
import time
import hashlib
import subprocess
import importlib
import importlib.metadata
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Keep the source pipeline's GPU visibility and numerical precision settings unchanged.


def notebook_source():
    """Capture readable source to create the self-contained frozen replay notebook."""
    try:
        shell = get_ipython()
        text = shell.history_manager.input_hist_raw[-1]
        if '# CCD_READABLE_SINGLE_CELL_V1' in text and 'def run_notebook(' in text:
            return text
    except (NameError, AttributeError, IndexError):
        pass
    if '__file__' in globals():
        text = Path(__file__).read_text(encoding='utf-8')
        if '# CCD_READABLE_SINGLE_CELL_V1' in text:
            return text
    raise RuntimeError('Run the entire cell at once so its source can be archived for reproduction.')


def ensure_dependencies(work_root, replay):
    """One pip transaction, only when requirements are missing or do not match."""
    from packaging.requirements import Requirement
    requirements = list(BOOTSTRAP_REQUIREMENTS)
    if replay:
        requirements = [f'{name}=={version}' for name, version in replay['packages'].items()]
        if list(sys.version_info[:2]) != replay['python_major_minor']:
            raise RuntimeError('Replay requires the same Python major/minor version as the saved run.')
    pending = []
    for text in requirements:
        req = Requirement(text)
        try:
            installed = importlib.metadata.version(req.name)
        except importlib.metadata.PackageNotFoundError:
            installed = None
        if installed is None or (req.specifier and not req.specifier.contains(installed, prereleases=True)):
            pending.append(text)
    if not pending:
        print('[setup] Required packages are already installed; no reinstall.')
        return
    # Track packages that may already be imported by the notebook environment itself.
    tracked = {'numpy': 'numpy', 'pandas': 'pandas', 'pyarrow': 'pyarrow', 'torch': 'torch',
               'transformers': 'transformers', 'datasets': 'datasets', 'unsloth': 'unsloth'}
    before = {}
    for module, package in tracked.items():
        if module in sys.modules:
            try:
                before[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                pass
    command = [
        sys.executable, '-m', 'pip', 'install',
        '--disable-pip-version-check',
        '--prefer-binary',
        '--upgrade-strategy', 'only-if-needed',
    ]

    # Kaggle currently exposes a distro-managed /usr/bin Python on some images.
    # PEP 668 blocks pip writes there unless this opt-in is supplied. Add it only
    # when the installed pip actually supports the option.
    pip_help = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--help'],
        capture_output=True, text=True, check=False,
    )
    if '--break-system-packages' in (pip_help.stdout + pip_help.stderr):
        command.append('--break-system-packages')

    if replay:
        # The saved lock includes the complete runtime dependency closure.
        lock = work_root / 'replay_requirements.txt'
        lock.write_text('\n'.join(requirements) + '\n', encoding='utf-8')
        command += ['--no-deps', '-r', str(lock)]
    else:
        # Submit the mutually compatible stack in one transaction.
        command += requirements

    print('[setup] Installing missing/incompatible packages before Unsloth imports.', flush=True)
    proc = subprocess.run(command, text=True, capture_output=True, check=False)
    if proc.stdout.strip():
        print(proc.stdout)
    if proc.returncode != 0:
        if proc.stderr.strip():
            print(proc.stderr, file=sys.stderr)
        raise RuntimeError(
            'Dependency installation failed. The full pip error is printed immediately above. '
            'Make sure Kaggle Internet is ON and run in a fresh session.'
        )

    importlib.invalidate_caches()
    stale = [name for name, version in before.items() if importlib.metadata.version(name) != version]
    if stale:
        raise RuntimeError('Installed packages changed libraries already loaded by this kernel: '
                           + ', '.join(stale) + '. Restart the Kaggle session and run this same cell again.')
    for text in requirements:
        req = Requirement(text)
        version = importlib.metadata.version(req.name)
        if req.specifier and not req.specifier.contains(version, prereleases=True):
            raise RuntimeError(f'Installation did not satisfy {text}; found {version}.')


def runtime_packages():
    """Freeze used packages and transitive dependencies, excluding unrelated Kaggle apps."""
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    todo = [(Requirement(req).name, frozenset()) for req in BOOTSTRAP_REQUIREMENTS]
    todo.extend([('torch', frozenset()), ('packaging', frozenset())])
    seen, versions = set(), {}
    while todo:
        name, extras = todo.pop()
        name = canonicalize_name(name)
        if (name, extras) in seen:
            continue
        seen.add((name, extras))
        dist = importlib.metadata.distribution(name)
        versions[name] = dist.version
        for text in dist.requires or []:
            req = Requirement(text)
            if req.marker and not any(req.marker.evaluate({'extra': x}) for x in {'', *extras}):
                continue
            actual = importlib.metadata.version(req.name)
            if req.specifier and not req.specifier.contains(actual, prereleases=True):
                raise RuntimeError(f'Runtime dependency conflict: {name} requires {req}, found {actual}.')
            todo.append((req.name, frozenset(req.extras)))
    return dict(sorted(versions.items()))


def gpu_signature():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU accelerator in Kaggle, then run this cell.')
    if torch.cuda.device_count() != 1:
        raise RuntimeError(
            'This protocol requires exactly one visible GPU so per-device batch 1 × '
            'gradient accumulation 16 remains effective batch size 16. '
            'Start a fresh Kaggle session and run this notebook again.'
        )
    return {'torch_cuda': torch.version.cuda,
            'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
            'capabilities': [list(torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]}


# -----------------------------
# 2) Authoritative ZIP configuration (unchanged bytes)
# -----------------------------
SOURCE_CONFIG_TEXT = r'''{
  "experiment_name": "ccd_uniform_v1",
  "data_seed": 42,
  "training_seeds": [42],
  "n_train": 300,
  "n_eval": 100,
  "n_demo_pool": 5,
  "max_seq_length": 2048,
  "system_instruction": "You are a helpful assistant that follows instructions exactly.",
  "training": {
    "num_train_epochs": 3.0,
    "learning_rate": 0.0001,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "warmup_ratio": 0.03,
    "lr_scheduler_type": "cosine",
    "weight_decay": 0.0,
    "max_grad_norm": 1.0,
    "optim": "adamw_8bit",
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ]
  },
  "generation": {
    "batch_size": 8,
    "self_consistency_temperature": 1.0,
    "self_consistency_top_p": 0.95,
    "self_consistency_top_k": 20,
    "configurations": [
      "zero",
      "cot",
      "fs0",
      "fs1",
      "fs3",
      "fs5",
      "sc1",
      "sc3",
      "sc5"
    ]
  },
  "execution": {
    "timeout_seconds": 5,
    "memory_limit_mb": 1024
  },
  "datasets": {
    "gsm8k": {
      "path": "openai/gsm8k",
      "subset": "main",
      "revision": null,
      "train_split": "train",
      "eval_split": "test",
      "max_new_tokens": 160
    },
    "strategyqa": {
      "path": "ChilleD/StrategyQA",
      "subset": null,
      "revision": null,
      "train_split": "train",
      "eval_split": "test",
      "max_new_tokens": 40
    },
    "logiqa": {
      "path": "lucasmccabe/logiqa",
      "subset": null,
      "revision": null,
      "train_split": "train",
      "eval_split": "test",
      "max_new_tokens": 96
    },
    "mbpp": {
      "path": "google-research-datasets/mbpp",
      "subset": "full",
      "revision": null,
      "train_split": "train",
      "eval_split": "test",
      "max_new_tokens": 384
    }
  },
  "models": {
    "gemma2_2b": "unsloth/gemma-2-2b-it-bnb-4bit",
    "gemma3_270m": "unsloth/gemma-3-270m-it-bnb-4bit",
    "gemma3_1b": "unsloth/gemma-3-1b-it-bnb-4bit",
    "llama3_8b": "unsloth/llama-3-8b-Instruct-bnb-4bit",
    "llama31_8b": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "llama32_1b": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "llama32_3b": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    "qwen25_05b": "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    "qwen25_15b": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "qwen25_7b": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "qwen3_06b": "unsloth/Qwen3-0.6B-unsloth-bnb-4bit",
    "qwen3_17b": "unsloth/Qwen3-1.7B-unsloth-bnb-4bit",
    "qwen35_08b": "unsloth/Qwen3.5-0.8B",
    "qwen35_2b": "unsloth/Qwen3.5-2B",
    "qwen35_4b": "unsloth/Qwen3.5-4B"
  }
}
'''
SOURCE_CONFIG_SHA256 = '7803c88f020b1d05b2a5942e2c075db9ca693d6a21bd781ba7657fe6abbdbf22'
BOOTSTRAP_REQUIREMENTS = ['numpy>=1.26', 'pandas>=2.2', 'datasets>=2.18', 'pyarrow', 'accelerate>=0.30', 'bitsandbytes>=0.45.5', 'sentencepiece', 'peft>=0.18.0', 'transformers==5.5.0', 'trl==0.24.0', 'unsloth==2026.8.19', 'unsloth_zoo==2026.8.13', 'torchvision', 'pillow', 'matplotlib', 'packaging']
# -----------------------------
# 3–10) Original ZIP pipeline implementation
# -----------------------------
#!/usr/bin/env python3
"""Uniform Compliance-Correctness Decomposition (CCD) experiment pipeline.

Commands
--------
prepare-manifests
    Download and normalize the four benchmarks once, then freeze the exact
    training, demonstration, and evaluation records shared by every worker.
assignments
    Create the complete model x dataset x seed job matrix.
run
    Run one independent model x dataset x seed block: pre-SFT evaluation,
    dataset-specific QLoRA training, and post-SFT evaluation.
validate
    Validate one completed run folder.
aggregate
    Validate and aggregate all completed run folders.
freeze-environment
    Save exact installed package versions for distribution to workers.

The script intentionally imports Unsloth before Transformers inside the run
command. Dataset preparation and aggregation do not require a GPU.
"""


import argparse
import ast
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import tempfile
import time
from collections import Counter
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple


PIPELINE_VERSION = "1.0.0"
EXPECTED_CONFIGS = ("zero", "cot", "fs0", "fs1", "fs3", "fs5", "sc1", "sc3", "sc5")
CHOICES = ("A", "B", "C", "D")


# ---------------------------------------------------------------------------
# Generic utilities
# ---------------------------------------------------------------------------


def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def atomic_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, ensure_ascii=False, sort_keys=True)
        handle.write("\n")
    tmp.replace(path)


def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False, sort_keys=True) + "\n")
    tmp.replace(path)


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSONL at {path}:{line_number}: {exc}") from exc
    return rows


def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def stable_seed(*parts: Any) -> int:
    payload = "|".join(str(part) for part in parts)
    return int(hashlib.sha256(payload.encode("utf-8")).hexdigest()[:8], 16)


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def require_keys(mapping: Dict[str, Any], keys: Sequence[str], label: str) -> None:
    missing = [key for key in keys if key not in mapping]
    if missing:
        raise ValueError(f"{label} is missing required keys: {missing}")


def package_versions() -> Dict[str, Optional[str]]:
    packages = [
        "torch",
        "transformers",
        "datasets",
        "accelerate",
        "bitsandbytes",
        "peft",
        "trl",
        "unsloth",
        "unsloth_zoo",
        "numpy",
        "pandas",
        "pyarrow",
    ]
    versions: Dict[str, Optional[str]] = {}
    for name in packages:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def environment_snapshot() -> Dict[str, Any]:
    gpu = None
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            check=False,
            capture_output=True,
            text=True,
            timeout=10,
        )
        gpu = result.stdout.strip() or None
    except (FileNotFoundError, subprocess.SubprocessError):
        pass
    return {
        "pipeline_version": PIPELINE_VERSION,
        "python": sys.version,
        "platform": platform.platform(),
        "packages": package_versions(),
        "gpu": gpu,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }


def validate_config(config: Dict[str, Any]) -> None:
    require_keys(
        config,
        ["experiment_name", "data_seed", "training_seeds", "n_train", "n_eval", "n_demo_pool", "training", "generation", "datasets", "models"],
        "config",
    )
    if int(config["n_eval"]) != 100:
        raise ValueError("Uniform CCD v1 requires n_eval=100 for both pre- and post-SFT.")
    actual = tuple(config["generation"]["configurations"])
    if actual != EXPECTED_CONFIGS:
        raise ValueError(f"Configurations must be exactly {EXPECTED_CONFIGS}; found {actual}")
    if int(config["n_train"]) <= int(config["n_demo_pool"]):
        raise ValueError("n_train must exceed n_demo_pool.")


# ---------------------------------------------------------------------------
# Dataset normalization and manifest preparation
# ---------------------------------------------------------------------------


def normalize_number(value: Any) -> str:
    from decimal import Decimal, InvalidOperation

    text = str(value).strip().replace(",", "")
    text = re.sub(r"\.$", "", text)
    try:
        number = Decimal(text)
        if number == number.to_integral_value():
            return str(number.quantize(Decimal(1)))
        return format(number.normalize(), "f")
    except InvalidOperation:
        return text


def gsm_gold(answer: Any) -> Tuple[str, str]:
    text = str(answer)
    if "####" in text:
        rationale, final = text.rsplit("####", 1)
        return rationale.strip(), normalize_number(final)
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    return "", normalize_number(numbers[-1]) if numbers else normalize_number(text)


def yes_no(value: Any) -> str:
    if isinstance(value, bool):
        return "Yes" if value else "No"
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return "Yes" if int(value) == 1 else "No"
    text = str(value).strip().lower()
    if text in {"yes", "y", "true", "1"}:
        return "Yes"
    if text in {"no", "n", "false", "0"}:
        return "No"
    raise ValueError(f"Cannot normalize yes/no label: {value!r}")


def choice_label(value: Any) -> str:
    if isinstance(value, bool):
        raise ValueError(f"Boolean is not a valid option label: {value!r}")
    if isinstance(value, int):
        if 0 <= value < 4:
            return CHOICES[value]
        if 1 <= value <= 4:
            return CHOICES[value - 1]
    text = str(value).strip().upper()
    if text in CHOICES:
        return text
    if text.isdigit():
        number = int(text)
        if 0 <= number < 4:
            return CHOICES[number]
        if 1 <= number <= 4:
            return CHOICES[number - 1]
    match = re.fullmatch(r"(?:OPTION\s*)?([ABCD])", text)
    if match:
        return match.group(1)
    raise ValueError(f"Cannot normalize multiple-choice label: {value!r}")


def source_id(dataset: str, split: str, row: Dict[str, Any], index: int) -> str:
    for key in ("qid", "task_id", "id", "idx", "index"):
        if key in row and row[key] is not None:
            return f"{dataset}:{split}:{row[key]}"
    identity = canonical_json({"dataset": dataset, "split": split, "index": index, "row": row})
    return f"{dataset}:{split}:sha256:{sha256_text(identity)[:20]}"


def canonicalize_example(dataset: str, split: str, row: Dict[str, Any], index: int) -> Dict[str, Any]:
    item_id = source_id(dataset, split, row, index)
    if dataset == "gsm8k":
        rationale, gold = gsm_gold(row["answer"])
        item = {
            "item_id": item_id,
            "question": str(row["question"]),
            "gold": gold,
            "reference_rationale": rationale,
        }
    elif dataset == "strategyqa":
        item = {
            "item_id": item_id,
            "question": str(row.get("question", row.get("query", ""))),
            "gold": yes_no(row.get("answer", row.get("label"))),
        }
    elif dataset == "logiqa":
        context = str(row.get("context", row.get("passage", ""))).strip()
        question = str(row.get("question", row.get("query", ""))).strip()
        options = list(row.get("options", row.get("answers", row.get("choices", []))))
        if len(options) != 4:
            raise ValueError(f"LogiQA item {item_id} has {len(options)} options, expected 4.")
        item = {
            "item_id": item_id,
            "context": context,
            "question": question,
            "options": [str(option) for option in options],
            "gold": choice_label(row.get("correct_option", row.get("label", row.get("answer")))),
        }
    elif dataset == "mbpp":
        tests = row.get("test_list", row.get("tests", []))
        if isinstance(tests, str):
            tests = [tests]
        challenge = row.get("challenge_test_list", [])
        if isinstance(challenge, str):
            challenge = [challenge]
        item = {
            "item_id": item_id,
            "question": str(row.get("text", row.get("prompt", row.get("description", "")))),
            "gold": "EXECUTION",
            "reference_code": str(row.get("code", row.get("solution", ""))),
            "test_setup_code": str(row.get("test_setup_code", "")),
            "tests": [str(test) for test in tests],
            "challenge_tests": [str(test) for test in challenge],
        }
        if not item["tests"]:
            raise ValueError(f"MBPP item {item_id} contains no tests.")
    else:
        raise ValueError(f"Unsupported dataset: {dataset}")

    if not item.get("question", "").strip():
        raise ValueError(f"Empty question in {item_id}")
    item["content_hash"] = sha256_text(canonical_json(item))
    return item


def load_hf_dataset(spec: Dict[str, Any]):
    from datasets import load_dataset

    kwargs: Dict[str, Any] = {}
    if spec.get("revision"):
        kwargs["revision"] = spec["revision"]
    elif spec.get("path") == "lucasmccabe/logiqa":
        # Compatibility-only source ref: current Datasets versions no longer
        # execute Hub dataset scripts. This HF-generated parquet ref contains
        # the same LogiQA splits/fields without executing logiqa.py.
        kwargs["revision"] = "refs/convert/parquet"
    if spec.get("subset"):
        return load_dataset(spec["path"], spec["subset"], **kwargs)
    return load_dataset(spec["path"], **kwargs)


def deterministic_select(records: List[Dict[str, Any]], count: int, seed: int) -> List[Dict[str, Any]]:
    if len(records) < count:
        raise ValueError(f"Requested {count} records but only {len(records)} are available.")
    order = list(range(len(records)))
    random.Random(seed).shuffle(order)
    return [records[index] for index in order[:count]]


def prepare_manifests(config_path: Path, manifest_dir: Path) -> None:
    config = load_json(config_path)
    validate_config(config)
    manifest_dir.mkdir(parents=True, exist_ok=True)
    config_hash = sha256_file(config_path)

    for dataset_name, spec in config["datasets"].items():
        output = manifest_dir / f"{dataset_name}.json"
        if output.exists():
            raise FileExistsError(f"Refusing to replace existing manifest: {output}")
        print(f"[manifest] Loading {dataset_name} from {spec['path']}...")
        raw = load_hf_dataset(spec)
        train_split = spec["train_split"]
        eval_split = spec["eval_split"]
        if train_split not in raw or eval_split not in raw:
            raise RuntimeError(
                f"{dataset_name} requires splits {train_split!r}/{eval_split!r}; available={list(raw.keys())}. "
                "The pipeline never falls back to training data for evaluation."
            )
        train_all = [
            canonicalize_example(dataset_name, train_split, dict(row), index)
            for index, row in enumerate(raw[train_split])
        ]
        eval_all = [
            canonicalize_example(dataset_name, eval_split, dict(row), index)
            for index, row in enumerate(raw[eval_split])
        ]
        required_train = int(config["n_train"]) + int(config["n_demo_pool"])
        selected_train = deterministic_select(train_all, required_train, stable_seed(config["data_seed"], dataset_name, "train"))
        training = selected_train[: int(config["n_train"])]
        demonstrations = selected_train[int(config["n_train"]) :]
        evaluation = deterministic_select(eval_all, int(config["n_eval"]), stable_seed(config["data_seed"], dataset_name, "eval"))

        train_ids = {row["item_id"] for row in training}
        demo_ids = {row["item_id"] for row in demonstrations}
        eval_ids = {row["item_id"] for row in evaluation}
        if len(train_ids) != len(training) or len(demo_ids) != len(demonstrations) or len(eval_ids) != len(evaluation):
            raise RuntimeError(f"Duplicate IDs detected while preparing {dataset_name}.")
        if train_ids & demo_ids or train_ids & eval_ids or demo_ids & eval_ids:
            raise RuntimeError(f"Split leakage detected while preparing {dataset_name}.")

        fingerprint = getattr(raw[train_split], "_fingerprint", None)
        eval_fingerprint = getattr(raw[eval_split], "_fingerprint", None)
        manifest = {
            "schema_version": 1,
            "pipeline_version": PIPELINE_VERSION,
            "dataset": dataset_name,
            "source": spec,
            "source_fingerprints": {"train": fingerprint, "eval": eval_fingerprint},
            "data_seed": config["data_seed"],
            "config_sha256": config_hash,
            "training": training,
            "demonstrations": demonstrations,
            "evaluation": evaluation,
        }
        manifest["manifest_payload_sha256"] = sha256_text(canonical_json(manifest))
        atomic_json(output, manifest)
        print(f"[manifest] Wrote {output} ({len(training)} train, {len(demonstrations)} demos, {len(evaluation)} eval).")


def create_assignments(config_path: Path, output_csv: Path) -> None:
    config = load_json(config_path)
    validate_config(config)
    if output_csv.exists():
        raise FileExistsError(f"Refusing to replace existing assignment file: {output_csv}")
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    rows = []
    for model_key in config["models"]:
        for dataset in config["datasets"]:
            for seed in config["training_seeds"]:
                rows.append(
                    {
                        "job_id": f"{model_key}__{dataset}__seed{seed}",
                        "model_key": model_key,
                        "dataset": dataset,
                        "seed": seed,
                        "assigned_to": "",
                        "status": "pending",
                    }
                )
    with output_csv.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {len(rows)} assignments to {output_csv}")


# ---------------------------------------------------------------------------
# Prompt construction
# ---------------------------------------------------------------------------


def display_question(dataset: str, item: Dict[str, Any]) -> str:
    if dataset == "logiqa":
        options = "\n".join(f"{letter}. {text}" for letter, text in zip(CHOICES, item["options"]))
        return f"Passage:\n{item['context']}\n\nQuestion:\n{item['question']}\n\nOptions:\n{options}"
    return item["question"]


def contract_instruction(dataset: str) -> str:
    if dataset == "gsm8k":
        return "Final Answer: <number>"
    if dataset == "strategyqa":
        return "Final Answer: Yes\nor\nFinal Answer: No"
    if dataset == "logiqa":
        return "Final Answer: <A, B, C, or D>"
    if dataset == "mbpp":
        return "Final Answer:\n```python\n<complete Python solution>\n```"
    raise ValueError(dataset)


def task_instruction(dataset: str) -> str:
    return {
        "gsm8k": "Solve the math problem.",
        "strategyqa": "Answer the question with Yes or No.",
        "logiqa": "Select the correct option.",
        "mbpp": "Write a complete Python solution for the programming task.",
    }[dataset]


def reference_answer(dataset: str, item: Dict[str, Any], include_reasoning: bool = False) -> str:
    if dataset == "gsm8k":
        if include_reasoning and item.get("reference_rationale"):
            return f"Reasoning: {item['reference_rationale']}\nFinal Answer: {item['gold']}"
        return f"Final Answer: {item['gold']}"
    if dataset in {"strategyqa", "logiqa"}:
        return f"Final Answer: {item['gold']}"
    if dataset == "mbpp":
        return f"Final Answer:\n```python\n{item['reference_code'].strip()}\n```"
    raise ValueError(dataset)


def evaluation_user_prompt(dataset: str, item: Dict[str, Any], configuration: str, demos: Sequence[Dict[str, Any]]) -> str:
    question = display_question(dataset, item)
    contract = contract_instruction(dataset)
    if configuration == "zero":
        return f"{task_instruction(dataset)}\nReturn ONLY the final result exactly as:\n{contract}\n\n{question}"
    if configuration in {"cot", "sc1", "sc3", "sc5"}:
        return f"{task_instruction(dataset)}\nGive brief reasoning, then finish exactly as:\n{contract}\n\n{question}"
    if configuration.startswith("fs"):
        k = int(configuration[2:])
        selected = list(demos[:k])
        example_text = ""
        for demo in selected:
            example_text += f"{display_question(dataset, demo)}\n{reference_answer(dataset, demo)}\n\n"
        return (
            f"{task_instruction(dataset)}\nFollow the examples and finish exactly as:\n{contract}\n\n"
            f"Examples:\n{example_text}Now solve:\n{question}"
        )
    raise ValueError(f"Unknown configuration: {configuration}")


def training_user_prompt(dataset: str, item: Dict[str, Any]) -> str:
    return (
        f"{task_instruction(dataset)}\nGive brief reasoning when appropriate, then finish exactly as:\n"
        f"{contract_instruction(dataset)}\n\n{display_question(dataset, item)}"
    )


# ---------------------------------------------------------------------------
# Parsing, voting, and code execution
# ---------------------------------------------------------------------------


FINAL_LINE = re.compile(r"(?im)^\s*Final\s+Answer\s*:\s*(.*?)\s*$")
FINAL_BOUNDARY = re.compile(r"(?im)^\s*Final\s+Answer\s*:\s*")


def unique_yes_no(text: str) -> Optional[str]:
    labels = {match.group(1).lower() for match in re.finditer(r"\b(yes|no)\b", text, flags=re.I)}
    if labels == {"yes"}:
        return "Yes"
    if labels == {"no"}:
        return "No"
    return None


def unique_choice(text: str) -> Optional[str]:
    labels = {match.group(1).upper() for match in re.finditer(r"\b([ABCD])\b", text, flags=re.I)}
    return next(iter(labels)) if len(labels) == 1 else None


def parse_number(text: str) -> Optional[str]:
    matches = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text)
    return normalize_number(matches[-1]) if matches else None


def strip_code_fences(text: str) -> str:
    text = text.strip()
    fenced = re.findall(r"```(?:python)?\s*(.*?)```", text, flags=re.I | re.S)
    if fenced:
        return fenced[-1].strip()
    return re.sub(r"^```(?:python)?\s*|\s*```$", "", text, flags=re.I | re.S).strip()


def normalize_code_vote(code: str) -> str:
    code = code.strip()
    if not code:
        return "__UNPARSEABLE__"
    try:
        return "AST:" + ast.dump(ast.parse(code), annotate_fields=False, include_attributes=False)
    except SyntaxError:
        return "SRC:" + "\n".join(line.rstrip() for line in code.splitlines()).strip()


def parse_candidate(dataset: str, completion: str) -> Dict[str, Any]:
    lines = FINAL_LINE.findall(completion)
    strict_region = lines[-1] if lines else ""
    if dataset == "gsm8k":
        strict_pred = parse_number(strict_region) if lines else None
        relaxed_pred = strict_pred if strict_pred is not None else parse_number(completion)
        format_ok = int(strict_pred is not None)
        pred = strict_pred if strict_pred is not None else relaxed_pred
        return {"pred": pred, "format_ok": format_ok, "vote_key": pred or "__UNPARSEABLE__", "code": None}
    if dataset == "strategyqa":
        strict_pred = unique_yes_no(strict_region) if lines else None
        relaxed_pred = strict_pred if strict_pred is not None else unique_yes_no(completion)
        format_ok = int(strict_pred is not None)
        pred = strict_pred if strict_pred is not None else relaxed_pred
        return {"pred": pred, "format_ok": format_ok, "vote_key": pred or "__UNPARSEABLE__", "code": None}
    if dataset == "logiqa":
        strict_pred = unique_choice(strict_region) if lines else None
        relaxed_pred = strict_pred if strict_pred is not None else unique_choice(completion)
        format_ok = int(strict_pred is not None)
        pred = strict_pred if strict_pred is not None else relaxed_pred
        return {"pred": pred, "format_ok": format_ok, "vote_key": pred or "__UNPARSEABLE__", "code": None}
    if dataset == "mbpp":
        boundaries = list(FINAL_BOUNDARY.finditer(completion))
        region = completion[boundaries[-1].end() :] if boundaries else completion
        code = strip_code_fences(region)
        format_ok = int(bool(boundaries) and bool(code.strip()))
        return {"pred": None, "format_ok": format_ok, "vote_key": normalize_code_vote(code), "code": code}
    raise ValueError(dataset)


def select_vote(candidates: List[Dict[str, Any]]) -> Tuple[int, str]:
    keys = [candidate["vote_key"] for candidate in candidates]
    counts = Counter(keys)
    maximum = max(counts.values())
    tied = {key for key, count in counts.items() if count == maximum}
    winner = next(key for key in keys if key in tied)
    supporters = [index for index, candidate in enumerate(candidates) if candidate["vote_key"] == winner]
    compliant = [index for index in supporters if candidates[index]["format_ok"] == 1]
    selected_index = compliant[0] if compliant else supporters[0]
    return selected_index, winner


def _execution_preexec(memory_limit_mb: int):
    def apply_limits() -> None:
        try:
            import resource

            memory = int(memory_limit_mb) * 1024 * 1024
            resource.setrlimit(resource.RLIMIT_AS, (memory, memory))
            resource.setrlimit(resource.RLIMIT_CPU, (10, 10))
            resource.setrlimit(resource.RLIMIT_FSIZE, (10 * 1024 * 1024, 10 * 1024 * 1024))
            resource.setrlimit(resource.RLIMIT_NOFILE, (64, 64))
        except (ImportError, ValueError, OSError):
            pass

    return apply_limits


def execute_mbpp(code: str, item: Dict[str, Any], timeout_seconds: int, memory_limit_mb: int) -> Dict[str, Any]:
    if not code.strip():
        return {"passed": 0, "status": "missing_code", "error_class": "MissingCode", "stderr": ""}
    program = "\n".join(
        part for part in [item.get("test_setup_code", ""), code, "\n".join(item["tests"])] if part.strip()
    )
    with tempfile.TemporaryDirectory(prefix="ccd_mbpp_") as directory:
        path = Path(directory) / "candidate.py"
        path.write_text(program, encoding="utf-8")
        try:
            result = subprocess.run(
                [sys.executable, "-I", str(path)],
                cwd=directory,
                env={"PYTHONHASHSEED": "0", "PATH": os.environ.get("PATH", "")},
                capture_output=True,
                text=True,
                timeout=timeout_seconds,
                check=False,
                preexec_fn=_execution_preexec(memory_limit_mb) if os.name == "posix" else None,
            )
        except subprocess.TimeoutExpired:
            return {"passed": 0, "status": "timeout", "error_class": "Timeout", "stderr": ""}
        stderr = result.stderr[-4000:]
        if result.returncode == 0:
            return {"passed": 1, "status": "passed", "error_class": None, "stderr": stderr}
        match = re.findall(r"([A-Za-z_][A-Za-z0-9_]*(?:Error|Exception))(?::|$)", stderr)
        error_class = match[-1] if match else "AssertionFailure" if "AssertionError" in stderr else "ExecutionFailure"
        return {"passed": 0, "status": "failed", "error_class": error_class, "stderr": stderr}


# ---------------------------------------------------------------------------
# Model/tokenizer helpers and generation
# ---------------------------------------------------------------------------


class TextInterface:
    def __init__(self, tokenizer_or_processor: Any, system_instruction: str, max_seq_length: int):
        self.outer = tokenizer_or_processor
        self.tokenizer = getattr(tokenizer_or_processor, "tokenizer", tokenizer_or_processor)
        self.system_instruction = system_instruction
        self.max_seq_length = int(max_seq_length)
        for obj in (self.outer, self.tokenizer):
            if hasattr(obj, "padding_side"):
                obj.padding_side = "left"
            if hasattr(obj, "truncation_side"):
                obj.truncation_side = "left"
            if getattr(obj, "pad_token_id", None) is None and getattr(obj, "eos_token_id", None) is not None:
                obj.pad_token_id = obj.eos_token_id

    @property
    def pad_token_id(self) -> Optional[int]:
        return getattr(self.tokenizer, "pad_token_id", getattr(self.outer, "pad_token_id", None))

    @property
    def eos_token_id(self) -> Optional[int]:
        return getattr(self.tokenizer, "eos_token_id", getattr(self.outer, "eos_token_id", None))

    def render(self, user_text: str, assistant_text: Optional[str] = None, generation: bool = False) -> str:
        from jinja2.exceptions import TemplateError

        messages = [
            {"role": "system", "content": self.system_instruction},
            {"role": "user", "content": user_text},
        ]
        if assistant_text is not None:
            messages.append({"role": "assistant", "content": assistant_text})

        # Some instruction-tuned checkpoints (notably certain Gemma templates)
        # reject an explicit system role. Their documented equivalent is to
        # prepend the same system instruction to the first user turn.
        system_folded_messages = [
            {
                "role": "user",
                "content": self.system_instruction + "\n\n" + user_text,
            }
        ]
        if assistant_text is not None:
            system_folded_messages.append(
                {"role": "assistant", "content": assistant_text}
            )

        def apply_template(obj: Any, msgs: List[Dict[str, str]]) -> str:
            kwargs = {"tokenize": False, "add_generation_prompt": generation}
            try:
                return obj.apply_chat_template(
                    msgs, enable_thinking=False, **kwargs
                )
            except TypeError:
                # Models/templates that do not expose enable_thinking.
                return obj.apply_chat_template(msgs, **kwargs)

        # Prefer the outer processor first. This is required by processor-backed
        # models such as Qwen3.5. For ordinary text models outer == tokenizer.
        seen = set()
        for obj in (self.outer, self.tokenizer):
            if id(obj) in seen:
                continue
            seen.add(id(obj))
            if not hasattr(obj, "apply_chat_template"):
                continue

            try:
                return apply_template(obj, messages)
            except TemplateError as exc:
                error_text = str(exc).lower()
                system_role_error = (
                    "system role not supported" in error_text
                    or "only user and assistant roles are supported" in error_text
                    or "only user and model roles are supported" in error_text
                )
                if not system_role_error:
                    raise
                # Compatibility-only fallback: preserve the exact instruction
                # text, but encode it in the first user turn because this
                # checkpoint's own chat template has no system-role channel.
                return apply_template(obj, system_folded_messages)

        raise RuntimeError("Tokenizer/processor has no chat template.")

    def encode(self, text_or_texts: Any, **kwargs: Any) -> Any:
        try:
            return self.tokenizer(text=text_or_texts, **kwargs)
        except TypeError:
            return self.tokenizer(text_or_texts, **kwargs)

    def decode(self, ids: Any) -> str:
        return self.tokenizer.decode(ids, skip_special_tokens=True)

    def token_count(self, text: str) -> int:
        encoded = self.encode(text, add_special_tokens=False, truncation=False)
        return len(encoded["input_ids"])


def cleanup_cuda() -> None:
    import gc

    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except ImportError:
        pass


# Infrastructure-only change: use the frozen resolution of the original model repository.
# Training, quantization and generation settings below remain the source settings.
def load_base_model(model_id: str, max_seq_length: int):
    # Unsloth must be imported before Transformers/PEFT training classes.
    import unsloth  # noqa: F401
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_REVISIONS[model_id]["repo_id"],
        revision=MODEL_REVISIONS[model_id]["revision"],
        use_exact_model_name=True,
        max_seq_length=max_seq_length,
        load_in_4bit=True,
        load_in_16bit=False,
        full_finetuning=False,
        fast_inference=False,
    )
    return model, tokenizer, FastLanguageModel


def generate_batch(
    model: Any,
    text: TextInterface,
    prompts: List[str],
    max_new_tokens: int,
    do_sample: bool,
    temperature: float,
    top_p: float,
    top_k: int,
    seed: int,
) -> Tuple[List[str], List[int], List[int], List[int]]:
    import torch

    set_global_seed(seed)
    model.eval()
    device = next(model.parameters()).device
    original_lengths = [text.token_count(prompt) for prompt in prompts]
    batch = text.encode(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=text.max_seq_length,
        add_special_tokens=False,
    ).to(device)
    padded_input_length = int(batch["input_ids"].shape[1])
    used_lengths = [int(value) for value in batch["attention_mask"].sum(dim=1).tolist()]
    kwargs: Dict[str, Any] = {
        **batch,
        "max_new_tokens": int(max_new_tokens),
        "do_sample": do_sample,
        "pad_token_id": text.pad_token_id,
        "eos_token_id": text.eos_token_id,
        "use_cache": True,
        "repetition_penalty": 1.0,
    }
    if do_sample:
        kwargs.update(temperature=float(temperature), top_p=float(top_p), top_k=int(top_k))
    with torch.inference_mode():
        output = model.generate(**kwargs)
    completions: List[str] = []
    generated_counts: List[int] = []
    for row in output:
        generated = row[padded_input_length:]
        if text.eos_token_id is not None:
            eos_positions = (generated == text.eos_token_id).nonzero(as_tuple=False)
            if eos_positions.numel() > 0:
                generated = generated[: int(eos_positions[0].item())]
        if text.pad_token_id is not None:
            generated = generated[generated != text.pad_token_id]
        completions.append(text.decode(generated))
        generated_counts.append(int(generated.numel()))
    return completions, generated_counts, original_lengths, used_lengths


def chunks(values: Sequence[Any], size: int) -> Iterable[Tuple[int, Sequence[Any]]]:
    for start in range(0, len(values), size):
        yield start, values[start : start + size]


def evaluate_configuration(
    model: Any,
    text: TextInterface,
    dataset: str,
    items: List[Dict[str, Any]],
    demos: List[Dict[str, Any]],
    configuration: str,
    stage: str,
    model_key: str,
    model_id: str,
    adapter_id: str,
    training_seed: int,
    config: Dict[str, Any],
    allow_code_execution: bool,
) -> List[Dict[str, Any]]:
    generation = config["generation"]
    dataset_spec = config["datasets"][dataset]
    batch_size = int(generation["batch_size"])
    prompts = [
        text.render(evaluation_user_prompt(dataset, item, configuration, demos), generation=True)
        for item in items
    ]
    if configuration.startswith("sc"):
        n_candidates = int(configuration[2:])
        do_sample = True
    else:
        n_candidates = 1
        do_sample = False

    candidates_by_item: List[List[Dict[str, Any]]] = [[] for _ in items]
    prompt_lengths_original = [0] * len(items)
    prompt_lengths_used = [0] * len(items)

    for candidate_index in range(n_candidates):
        for start, prompt_batch in chunks(prompts, batch_size):
            batch_seed = stable_seed(
                config["experiment_name"],
                dataset,
                configuration,
                candidate_index,
                start,
                training_seed,
            )
            completions, token_counts, original_lengths, used_lengths = generate_batch(
                model=model,
                text=text,
                prompts=list(prompt_batch),
                max_new_tokens=int(dataset_spec["max_new_tokens"]),
                do_sample=do_sample,
                temperature=float(generation["self_consistency_temperature"]),
                top_p=float(generation["self_consistency_top_p"]),
                top_k=int(generation["self_consistency_top_k"]),
                seed=batch_seed,
            )
            for offset, completion in enumerate(completions):
                item_index = start + offset
                parsed = parse_candidate(dataset, completion)
                candidates_by_item[item_index].append(
                    {
                        "candidate_index": candidate_index,
                        "sampling_seed": batch_seed if do_sample else None,
                        "completion": completion,
                        "completion_sha256": sha256_text(completion),
                        "generated_tokens": token_counts[offset],
                        "hit_token_ceiling": int(token_counts[offset] >= int(dataset_spec["max_new_tokens"])),
                        **parsed,
                    }
                )
                prompt_lengths_original[item_index] = original_lengths[offset]
                prompt_lengths_used[item_index] = used_lengths[offset]

    records: List[Dict[str, Any]] = []
    for index, item in enumerate(items):
        candidates = candidates_by_item[index]
        selected_index, winner_key = select_vote(candidates)
        selected = candidates[selected_index]
        any_compliant = int(any(candidate["format_ok"] == 1 for candidate in candidates))
        winner_supported_compliant = int(
            any(candidate["format_ok"] == 1 and candidate["vote_key"] == winner_key for candidate in candidates)
        )
        execution = None
        if dataset == "mbpp":
            if not allow_code_execution:
                raise RuntimeError(
                    "MBPP requires --allow-code-execution. Run only inside a disposable isolated environment; "
                    "the resource limits are not a complete security sandbox."
                )
            execution = execute_mbpp(
                selected.get("code") or "",
                item,
                timeout_seconds=int(config["execution"]["timeout_seconds"]),
                memory_limit_mb=int(config["execution"]["memory_limit_mb"]),
            )
            correctness = int(execution["passed"])
            functional_correctness: Optional[int] = correctness
        else:
            correctness = int(selected.get("pred") is not None and selected.get("pred") == item["gold"])
            functional_correctness = None
        strict_success = int(winner_supported_compliant == 1 and correctness == 1)
        record = {
            "schema_version": 1,
            "pipeline_version": PIPELINE_VERSION,
            "experiment_name": config["experiment_name"],
            "model_key": model_key,
            "model_id": model_id,
            "adapter_id": adapter_id,
            "training_seed": training_seed,
            "dataset": dataset,
            "item_id": item["item_id"],
            "item_content_hash": item["content_hash"],
            "configuration": configuration,
            "stage": stage,
            "prompt": prompts[index],
            "prompt_sha256": sha256_text(prompts[index]),
            "prompt_tokens_original": prompt_lengths_original[index],
            "prompt_tokens_used": prompt_lengths_used[index],
            "prompt_truncated": int(prompt_lengths_original[index] > prompt_lengths_used[index]),
            "gold": item["gold"],
            "winner_vote_key": winner_key,
            "selected_candidate_index": selected_index,
            "selected_prediction": selected.get("pred"),
            "selected_code": selected.get("code"),
            "format_ok": winner_supported_compliant,
            "any_sample_compliant": any_compliant,
            "relaxed_correctness": correctness,
            "functional_correctness": functional_correctness,
            "strict_success": strict_success,
            "execution": execution,
            "candidates": candidates,
        }
        records.append(record)
    return records


def summarize_records(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    if not records:
        raise ValueError("Cannot summarize empty records.")

    def mean(field: str) -> Optional[float]:
        values = [row[field] for row in records if row.get(field) is not None]
        return sum(values) / len(values) if values else None

    return {
        "n": len(records),
        "format_rate": mean("format_ok"),
        "strict_success_rate": mean("strict_success"),
        "relaxed_correctness_rate": mean("relaxed_correctness"),
        "functional_correctness_rate": mean("functional_correctness"),
        "valid_but_wrong_rate": mean("format_ok") - mean("strict_success"),
        "prompt_truncation_rate": mean("prompt_truncated"),
        "token_ceiling_rate": sum(
            int(any(candidate["hit_token_ceiling"] for candidate in row["candidates"])) for row in records
        )
        / len(records),
    }


# ---------------------------------------------------------------------------
# QLoRA training and one-block orchestration
# ---------------------------------------------------------------------------


class MaskedCollator:
    def __init__(self, pad_token_id: int):
        self.pad_token_id = int(pad_token_id)

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        import torch

        maximum = max(len(row["input_ids"]) for row in batch)
        input_ids, attention_mask, labels = [], [], []
        for row in batch:
            pad = maximum - len(row["input_ids"])
            input_ids.append(row["input_ids"] + [self.pad_token_id] * pad)
            attention_mask.append(row["attention_mask"] + [0] * pad)
            labels.append(row["labels"] + [-100] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


def train_adapter(
    base_model: Any,
    text: TextInterface,
    fast_language_model: Any,
    dataset_name: str,
    training_items: List[Dict[str, Any]],
    adapter_dir: Path,
    training_seed: int,
    config: Dict[str, Any],
) -> Tuple[Any, Dict[str, Any]]:
    # Unsloth is already imported. Import Trainer only now.
    from datasets import Dataset
    from transformers import Trainer, TrainingArguments

    training = config["training"]
    set_global_seed(training_seed)
    model = fast_language_model.get_peft_model(
        base_model,
        r=int(training["lora_r"]),
        target_modules=list(training["target_modules"]),
        lora_alpha=int(training["lora_alpha"]),
        lora_dropout=float(training["lora_dropout"]),
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=training_seed,
        max_seq_length=int(config["max_seq_length"]),
    )
    model.config.use_cache = False
    tokenized: List[Dict[str, Any]] = []
    audit_rows: List[Dict[str, Any]] = []
    for item in training_items:
        user = training_user_prompt(dataset_name, item)
        assistant = reference_answer(dataset_name, item, include_reasoning=True)
        prefix = text.render(user, generation=True)
        full = text.render(user, assistant_text=assistant, generation=False)
        prefix_ids = text.encode(prefix, add_special_tokens=False, truncation=False)["input_ids"]
        full_untruncated = text.encode(full, add_special_tokens=False, truncation=False)["input_ids"]
        full_ids = full_untruncated[-text.max_seq_length :]
        # Left truncation can remove part of the prefix. Recompute the masked
        # prefix length relative to the retained suffix.
        removed = max(0, len(full_untruncated) - len(full_ids))
        retained_prefix = max(0, min(len(prefix_ids) - removed, len(full_ids)))
        labels = list(full_ids)
        labels[:retained_prefix] = [-100] * retained_prefix
        assistant_tokens = sum(label != -100 for label in labels)
        if assistant_tokens <= 0:
            raise RuntimeError(f"No assistant tokens remain after truncation for {item['item_id']}")
        tokenized.append({"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels})
        audit_rows.append(
            {
                "item_id": item["item_id"],
                "full_tokens_original": len(full_untruncated),
                "full_tokens_used": len(full_ids),
                "truncated": int(len(full_untruncated) > len(full_ids)),
                "assistant_tokens": assistant_tokens,
            }
        )
    training_dataset = Dataset.from_list(tokenized)
    adapter_dir.mkdir(parents=True, exist_ok=True)
    arguments = TrainingArguments(
        output_dir=str(adapter_dir / "trainer"),
        per_device_train_batch_size=int(training["per_device_train_batch_size"]),
        gradient_accumulation_steps=int(training["gradient_accumulation_steps"]),
        learning_rate=float(training["learning_rate"]),
        warmup_ratio=float(training["warmup_ratio"]),
        lr_scheduler_type=str(training["lr_scheduler_type"]),
        num_train_epochs=float(training["num_train_epochs"]),
        weight_decay=float(training["weight_decay"]),
        max_grad_norm=float(training["max_grad_norm"]),
        optim=str(training["optim"]),
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        fp16=False,
        bf16=False,
        seed=training_seed,
        data_seed=training_seed,
        average_tokens_across_devices=False,
    )
    trainer = Trainer(
        model=model,
        args=arguments,
        train_dataset=training_dataset,
        data_collator=MaskedCollator(text.pad_token_id),
    )
    started = time.time()
    result = trainer.train()
    elapsed = time.time() - started
    model.save_pretrained(str(adapter_dir))
    try:
        text.outer.save_pretrained(str(adapter_dir))
    except Exception:
        text.tokenizer.save_pretrained(str(adapter_dir))
    audit = {
        "elapsed_seconds": elapsed,
        "train_result": result.metrics,
        "trainer_log_history": trainer.state.log_history,
        "n_training_items": len(training_items),
        "total_assistant_tokens_per_epoch": sum(row["assistant_tokens"] for row in audit_rows),
        "truncated_training_items": sum(row["truncated"] for row in audit_rows),
        "item_audit": audit_rows,
    }
    return model, audit


def run_experiment(
    config_path: Path,
    manifest_dir: Path,
    results_root: Path,
    model_key: str,
    dataset_name: str,
    training_seed: int,
    allow_code_execution: bool,
) -> Path:
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    config = load_json(config_path)
    validate_config(config)
    if model_key not in config["models"]:
        raise KeyError(f"Unknown model key {model_key!r}. Available: {list(config['models'])}")
    if dataset_name not in config["datasets"]:
        raise KeyError(f"Unknown dataset {dataset_name!r}. Available: {list(config['datasets'])}")
    if training_seed not in [int(seed) for seed in config["training_seeds"]]:
        raise ValueError(f"Seed {training_seed} is not declared in training_seeds={config['training_seeds']}")
    manifest_path = manifest_dir / f"{dataset_name}.json"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing frozen manifest: {manifest_path}")
    manifest = load_json(manifest_path)
    if len(manifest["training"]) != int(config["n_train"]):
        raise ValueError("Manifest training count does not match config.")
    if len(manifest["evaluation"]) != 100:
        raise ValueError("Manifest must contain exactly 100 evaluation items.")
    config_hash = sha256_file(config_path)
    if manifest.get("config_sha256") != config_hash:
        raise ValueError(
            "Frozen manifest was prepared from a different experiment_config.json. "
            "Use the coordinator's unchanged config and manifests."
        )
    manifest_hash = sha256_file(manifest_path)
    model_id = config["models"][model_key]
    adapter_id = f"{model_key}__{dataset_name}__seed{training_seed}"
    run_dir = results_root / adapter_id
    if run_dir.exists():
        raise FileExistsError(
            f"Run directory already exists: {run_dir}. The pipeline never overwrites results. "
            "Validate it or choose a new results root."
        )
    records_dir = run_dir / "records"
    adapter_dir = run_dir / "adapter"
    records_dir.mkdir(parents=True, exist_ok=False)
    atomic_json(run_dir / "RUNNING.json", {"started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
    atomic_json(run_dir / "environment.json", environment_snapshot())
    metadata = {
        "pipeline_version": PIPELINE_VERSION,
        "experiment_name": config["experiment_name"],
        "adapter_id": adapter_id,
        "model_key": model_key,
        "model_id": model_id,
        "dataset": dataset_name,
        "training_seed": training_seed,
        "config_sha256": config_hash,
        "manifest_sha256": manifest_hash,
        "n_train": len(manifest["training"]),
        "n_eval_pre": len(manifest["evaluation"]),
        "n_eval_post": len(manifest["evaluation"]),
        "configurations": config["generation"]["configurations"],
    }
    atomic_json(run_dir / "run_metadata.json", metadata)

    set_global_seed(training_seed)
    print(f"[run] Loading {model_key}: {model_id}")
    base_model, tokenizer, fast_language_model = load_base_model(model_id, int(config["max_seq_length"]))
    text = TextInterface(tokenizer, config["system_instruction"], int(config["max_seq_length"]))
    if getattr(base_model, "generation_config", None) is not None:
        base_model.generation_config.pad_token_id = text.pad_token_id
        base_model.generation_config.eos_token_id = text.eos_token_id

    summaries: Dict[str, Dict[str, Any]] = {"pre_sft": {}, "post_sft": {}}
    print("[run] Pre-SFT evaluation")
    try:
        fast_language_model.for_inference(base_model)
    except Exception:
        base_model.eval()
    for configuration in config["generation"]["configurations"]:
        print(f"[pre] {configuration}")
        records = evaluate_configuration(
            base_model,
            text,
            dataset_name,
            manifest["evaluation"],
            manifest["demonstrations"],
            configuration,
            "pre_sft",
            model_key,
            model_id,
            adapter_id,
            training_seed,
            config,
            allow_code_execution,
        )
        write_jsonl(records_dir / f"pre_sft__{configuration}.jsonl", records)
        summaries["pre_sft"][configuration] = summarize_records(records)
        cleanup_cuda()

    print("[run] Dataset-specific QLoRA training")
    trained_model, training_audit = train_adapter(
        base_model,
        text,
        fast_language_model,
        dataset_name,
        manifest["training"],
        adapter_dir,
        training_seed,
        config,
    )
    atomic_json(run_dir / "training_audit.json", training_audit)
    trained_model.config.use_cache = True
    try:
        fast_language_model.for_inference(trained_model)
    except Exception:
        trained_model.eval()

    print("[run] Post-SFT evaluation")
    for configuration in config["generation"]["configurations"]:
        print(f"[post] {configuration}")
        records = evaluate_configuration(
            trained_model,
            text,
            dataset_name,
            manifest["evaluation"],
            manifest["demonstrations"],
            configuration,
            "post_sft",
            model_key,
            model_id,
            adapter_id,
            training_seed,
            config,
            allow_code_execution,
        )
        write_jsonl(records_dir / f"post_sft__{configuration}.jsonl", records)
        summaries["post_sft"][configuration] = summarize_records(records)
        cleanup_cuda()
    atomic_json(run_dir / "stage_summaries.json", summaries)
    validation = validate_run(run_dir, expected_n=100, raise_on_error=True)
    atomic_json(
        run_dir / "run_complete.json",
        {
            "completed_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "validation": validation,
        },
    )
    (run_dir / "RUNNING.json").unlink(missing_ok=True)
    print(f"[run] Completed and validated: {run_dir}")
    return run_dir


# ---------------------------------------------------------------------------
# Validation and aggregation
# ---------------------------------------------------------------------------


def validate_run(run_dir: Path, expected_n: int = 100, raise_on_error: bool = False) -> Dict[str, Any]:
    errors: List[str] = []
    metadata_path = run_dir / "run_metadata.json"
    if not metadata_path.exists():
        errors.append("missing run_metadata.json")
        result = {"valid": False, "errors": errors}
        if raise_on_error:
            raise RuntimeError(canonical_json(result))
        return result
    metadata = load_json(metadata_path)
    record_dir = run_dir / "records"
    cell_checks: Dict[str, Any] = {}
    for configuration in metadata.get("configurations", EXPECTED_CONFIGS):
        pre_path = record_dir / f"pre_sft__{configuration}.jsonl"
        post_path = record_dir / f"post_sft__{configuration}.jsonl"
        if not pre_path.exists() or not post_path.exists():
            errors.append(f"missing pre/post records for {configuration}")
            continue
        pre = read_jsonl(pre_path)
        post = read_jsonl(post_path)
        pre_ids = [row.get("item_id") for row in pre]
        post_ids = [row.get("item_id") for row in post]
        if len(pre) != expected_n or len(post) != expected_n:
            errors.append(f"{configuration}: expected {expected_n}/{expected_n} rows, found {len(pre)}/{len(post)}")
        if len(set(pre_ids)) != len(pre_ids) or len(set(post_ids)) != len(post_ids):
            errors.append(f"{configuration}: duplicate item IDs")
        if pre_ids != post_ids:
            errors.append(f"{configuration}: pre/post item IDs or order differ")
        pre_prompts = {row["item_id"]: row.get("prompt_sha256") for row in pre}
        post_prompts = {row["item_id"]: row.get("prompt_sha256") for row in post}
        if pre_prompts != post_prompts:
            errors.append(f"{configuration}: pre/post prompts differ")
        pre_content = {row["item_id"]: row.get("item_content_hash") for row in pre}
        post_content = {row["item_id"]: row.get("item_content_hash") for row in post}
        if pre_content != post_content:
            errors.append(f"{configuration}: pre/post item content hashes differ")
        for stage, rows in (("pre", pre), ("post", post)):
            for row in rows:
                required = [
                    "prompt",
                    "candidates",
                    "format_ok",
                    "strict_success",
                    "relaxed_correctness",
                    "item_content_hash",
                ]
                missing = [key for key in required if key not in row]
                if missing:
                    errors.append(f"{configuration}/{stage}/{row.get('item_id')}: missing {missing}")
                    break
                if not isinstance(row["candidates"], list) or not row["candidates"]:
                    errors.append(f"{configuration}/{stage}/{row.get('item_id')}: empty candidates")
                    break
                if int(row["strict_success"]) > int(row["format_ok"]):
                    errors.append(f"{configuration}/{stage}/{row.get('item_id')}: S > F")
                    break
        cell_checks[configuration] = {
            "pre_rows": len(pre),
            "post_rows": len(post),
            "same_ids": pre_ids == post_ids,
            "same_prompts": pre_prompts == post_prompts,
        }
    result = {"valid": not errors, "errors": errors, "cells": cell_checks}
    if errors and raise_on_error:
        raise RuntimeError("Run validation failed:\n- " + "\n- ".join(errors))
    return result


def paired_cell_summary(pre: List[Dict[str, Any]], post: List[Dict[str, Any]], metadata: Dict[str, Any], configuration: str) -> Dict[str, Any]:
    pre_map = {row["item_id"]: row for row in pre}
    post_map = {row["item_id"]: row for row in post}
    if set(pre_map) != set(post_map):
        raise ValueError(f"Mismatched item IDs in {metadata['adapter_id']} / {configuration}")

    def rate(rows: Dict[str, Dict[str, Any]], field: str) -> Optional[float]:
        values = [row[field] for row in rows.values() if row.get(field) is not None]
        return sum(values) / len(values) if values else None

    pre_f, post_f = rate(pre_map, "format_ok"), rate(post_map, "format_ok")
    pre_s, post_s = rate(pre_map, "strict_success"), rate(post_map, "strict_success")
    pre_c, post_c = rate(pre_map, "relaxed_correctness"), rate(post_map, "relaxed_correctness")
    pre_func, post_func = rate(pre_map, "functional_correctness"), rate(post_map, "functional_correctness")
    delta_f = post_f - pre_f
    delta_s = post_s - pre_s
    return {
        "adapter_id": metadata["adapter_id"],
        "model_key": metadata["model_key"],
        "model_id": metadata["model_id"],
        "dataset": metadata["dataset"],
        "training_seed": metadata["training_seed"],
        "configuration": configuration,
        "n": len(pre_map),
        "pre_F": pre_f,
        "post_F": post_f,
        "delta_F": delta_f,
        "pre_S": pre_s,
        "post_S": post_s,
        "delta_S": delta_s,
        "pre_V": pre_f - pre_s,
        "post_V": post_f - post_s,
        "G": delta_f - delta_s,
        "pre_C_rel": pre_c,
        "post_C_rel": post_c,
        "delta_C_rel": post_c - pre_c,
        "pre_C_func": pre_func,
        "post_C_func": post_func,
        "delta_C_func": (post_func - pre_func) if pre_func is not None and post_func is not None else None,
    }


def mean_optional(values: Iterable[Optional[float]]) -> Optional[float]:
    present = [float(value) for value in values if value is not None and not math.isnan(float(value))]
    return sum(present) / len(present) if present else None


def percentile(values: List[float], p: float) -> float:
    if not values:
        raise ValueError("Cannot take percentile of empty list")
    ordered = sorted(values)
    position = (len(ordered) - 1) * p
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    return ordered[lower] * (upper - position) + ordered[upper] * (position - lower)


def model_cluster_bootstrap(blocks: List[Dict[str, Any]], field: str, repetitions: int = 10000, seed: int = 42) -> Optional[Dict[str, float]]:
    by_model: Dict[str, List[float]] = {}
    for block in blocks:
        value = block.get(field)
        if value is not None:
            by_model.setdefault(block["model_key"], []).append(float(value))
    models = sorted(by_model)
    if not models:
        return None
    model_means = {model: sum(values) / len(values) for model, values in by_model.items()}
    rng = random.Random(seed)
    estimates: List[float] = []
    for _ in range(repetitions):
        sampled = [rng.choice(models) for _ in models]
        estimates.append(sum(model_means[model] for model in sampled) / len(sampled))
    point = sum(model_means.values()) / len(model_means)
    return {"estimate": point, "ci_low": percentile(estimates, 0.025), "ci_high": percentile(estimates, 0.975)}


def write_csv(path: Path, rows: List[Dict[str, Any]]) -> None:
    if not rows:
        raise ValueError(f"No rows to write to {path}")
    path.parent.mkdir(parents=True, exist_ok=True)
    fields: List[str] = []
    for row in rows:
        for field in row:
            if field not in fields:
                fields.append(field)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def aggregate_results(results_root: Path, output_dir: Path, expected_n: int = 100) -> None:
    complete_markers = sorted(results_root.glob("*/run_complete.json"))
    if not complete_markers:
        raise FileNotFoundError(f"No completed runs found under {results_root}")
    if output_dir.exists():
        raise FileExistsError(f"Refusing to replace existing aggregation directory: {output_dir}")
    output_dir.mkdir(parents=True)
    cell_rows: List[Dict[str, Any]] = []
    validation_rows: List[Dict[str, Any]] = []
    for marker in complete_markers:
        run_dir = marker.parent
        validation = validate_run(run_dir, expected_n=expected_n, raise_on_error=False)
        validation_rows.append({"run_dir": str(run_dir), "valid": validation["valid"], "errors": " | ".join(validation["errors"])})
        if not validation["valid"]:
            continue
        metadata = load_json(run_dir / "run_metadata.json")
        for configuration in metadata["configurations"]:
            pre = read_jsonl(run_dir / "records" / f"pre_sft__{configuration}.jsonl")
            post = read_jsonl(run_dir / "records" / f"post_sft__{configuration}.jsonl")
            cell_rows.append(paired_cell_summary(pre, post, metadata, configuration))
    write_csv(output_dir / "validation.csv", validation_rows)
    invalid = [row for row in validation_rows if not row["valid"]]
    if invalid:
        raise RuntimeError(f"{len(invalid)} runs failed validation; inspect {output_dir / 'validation.csv'}")
    write_csv(output_dir / "cell_summary.csv", cell_rows)

    block_groups: Dict[Tuple[str, str, int], List[Dict[str, Any]]] = {}
    for row in cell_rows:
        key = (row["model_key"], row["dataset"], int(row["training_seed"]))
        block_groups.setdefault(key, []).append(row)
    metric_fields = ["delta_F", "delta_S", "G", "delta_C_rel", "delta_C_func"]
    block_rows: List[Dict[str, Any]] = []
    for (model_key, dataset, seed), rows in sorted(block_groups.items()):
        block = {
            "adapter_id": rows[0]["adapter_id"],
            "model_key": model_key,
            "dataset": dataset,
            "training_seed": seed,
            "n_configurations": len(rows),
        }
        for field in metric_fields:
            block[field] = mean_optional(row.get(field) for row in rows)
        block_rows.append(block)
    write_csv(output_dir / "block_summary.csv", block_rows)

    overall = {
        "n_completed_runs": len(block_rows),
        "n_cells": len(cell_rows),
        "equal_block_means": {field: mean_optional(row.get(field) for row in block_rows) for field in metric_fields},
        "positive_gap_blocks": sum(float(row["G"]) > 0 for row in block_rows),
        "model_cluster_bootstrap": {
            field: model_cluster_bootstrap(block_rows, field, repetitions=10000, seed=42) for field in metric_fields
        },
    }
    atomic_json(output_dir / "overall_summary.json", overall)
    print(f"Aggregated {len(block_rows)} blocks and {len(cell_rows)} cells into {output_dir}")


def freeze_environment(output: Path) -> None:
    if output.exists():
        raise FileExistsError(f"Refusing to replace existing lock file: {output}")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze", "--all"],
        check=True,
        capture_output=True,
        text=True,
    )
    output.write_text(result.stdout, encoding="utf-8")
    print(f"Wrote exact environment lock to {output}")


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = parser.add_subparsers(dest="command", required=True)

    manifests = sub.add_parser("prepare-manifests", help="Freeze exact dataset records once.")
    manifests.add_argument("--config", type=Path, default=Path("experiment_config.json"))
    manifests.add_argument("--manifest-dir", type=Path, default=Path("manifests"))

    assignments = sub.add_parser("assignments", help="Write model x dataset x seed job matrix.")
    assignments.add_argument("--config", type=Path, default=Path("experiment_config.json"))
    assignments.add_argument("--output", type=Path, default=Path("assignments.csv"))

    run = sub.add_parser("run", help="Run one model x dataset x seed block.")
    run.add_argument("--config", type=Path, default=Path("experiment_config.json"))
    run.add_argument("--manifest-dir", type=Path, default=Path("manifests"))
    run.add_argument("--results-root", type=Path, default=Path("results"))
    run.add_argument("--model", required=True, dest="model_key")
    run.add_argument("--dataset", required=True, choices=["gsm8k", "strategyqa", "logiqa", "mbpp"])
    run.add_argument("--seed", required=True, type=int)
    run.add_argument(
        "--allow-code-execution",
        action="store_true",
        help="Required for MBPP. Generated code is untrusted; use only in a disposable isolated runtime.",
    )

    validate = sub.add_parser("validate", help="Validate a completed run folder.")
    validate.add_argument("--run-dir", type=Path, required=True)
    validate.add_argument("--expected-n", type=int, default=100)

    aggregate = sub.add_parser("aggregate", help="Validate and aggregate all completed runs.")
    aggregate.add_argument("--results-root", type=Path, default=Path("results"))
    aggregate.add_argument("--output-dir", type=Path, default=Path("aggregate"))
    aggregate.add_argument("--expected-n", type=int, default=100)

    freeze = sub.add_parser("freeze-environment", help="Freeze exact package versions after a validated smoke run.")
    freeze.add_argument("--output", type=Path, default=Path("requirements-lock.txt"))
    return parser


def main() -> None:
    args = build_parser().parse_args()
    if args.command == "prepare-manifests":
        prepare_manifests(args.config, args.manifest_dir)
    elif args.command == "assignments":
        create_assignments(args.config, args.output)
    elif args.command == "run":
        run_experiment(
            args.config,
            args.manifest_dir,
            args.results_root,
            args.model_key,
            args.dataset,
            args.seed,
            args.allow_code_execution,
        )
    elif args.command == "validate":
        result = validate_run(args.run_dir, expected_n=args.expected_n, raise_on_error=False)
        print(json.dumps(result, indent=2))
        if not result["valid"]:
            raise SystemExit(1)
    elif args.command == "aggregate":
        aggregate_results(args.results_root, args.output_dir, expected_n=args.expected_n)
    elif args.command == "freeze-environment":
        freeze_environment(args.output)
    else:
        raise AssertionError(args.command)


def prepare_selected_manifest(config_path: Path, manifest_dir: Path, dataset_name: str, revision: str) -> None:
    config = load_json(config_path)
    validate_config(config)
    manifest_dir.mkdir(parents=True, exist_ok=True)
    config_hash = sha256_file(config_path)

    for dataset_name, spec in [(dataset_name, config["datasets"][dataset_name])]:
        output = manifest_dir / f"{dataset_name}.json"
        if output.exists():
            raise FileExistsError(f"Refusing to replace existing manifest: {output}")
        print(f"[manifest] Loading {dataset_name} from {spec['path']}...")
        raw = load_hf_dataset(dict(spec, revision=revision))
        train_split = spec["train_split"]
        eval_split = spec["eval_split"]
        if train_split not in raw or eval_split not in raw:
            raise RuntimeError(
                f"{dataset_name} requires splits {train_split!r}/{eval_split!r}; available={list(raw.keys())}. "
                "The pipeline never falls back to training data for evaluation."
            )
        train_all = [
            canonicalize_example(dataset_name, train_split, dict(row), index)
            for index, row in enumerate(raw[train_split])
        ]
        eval_all = [
            canonicalize_example(dataset_name, eval_split, dict(row), index)
            for index, row in enumerate(raw[eval_split])
        ]
        required_train = int(config["n_train"]) + int(config["n_demo_pool"])
        selected_train = deterministic_select(train_all, required_train, stable_seed(config["data_seed"], dataset_name, "train"))
        training = selected_train[: int(config["n_train"])]
        demonstrations = selected_train[int(config["n_train"]) :]
        evaluation = deterministic_select(eval_all, int(config["n_eval"]), stable_seed(config["data_seed"], dataset_name, "eval"))

        train_ids = {row["item_id"] for row in training}
        demo_ids = {row["item_id"] for row in demonstrations}
        eval_ids = {row["item_id"] for row in evaluation}
        if len(train_ids) != len(training) or len(demo_ids) != len(demonstrations) or len(eval_ids) != len(evaluation):
            raise RuntimeError(f"Duplicate IDs detected while preparing {dataset_name}.")
        if train_ids & demo_ids or train_ids & eval_ids or demo_ids & eval_ids:
            raise RuntimeError(f"Split leakage detected while preparing {dataset_name}.")

        fingerprint = getattr(raw[train_split], "_fingerprint", None)
        eval_fingerprint = getattr(raw[eval_split], "_fingerprint", None)
        manifest = {
            "schema_version": 1,
            "pipeline_version": PIPELINE_VERSION,
            "dataset": dataset_name,
            "source": spec,
            "source_fingerprints": {"train": fingerprint, "eval": eval_fingerprint},
            "data_seed": config["data_seed"],
            "config_sha256": config_hash,
            "training": training,
            "demonstrations": demonstrations,
            "evaluation": evaluation,
        }
        manifest["manifest_payload_sha256"] = sha256_text(canonical_json(manifest))
        atomic_json(output, manifest)
        print(f"[manifest] Wrote {output} ({len(training)} train, {len(demonstrations)} demos, {len(evaluation)} eval).")

# -----------------------------
# 11) Old-notebook output layout — reporting only, after CCD validation
# -----------------------------
def bootstrap_ci(flags, n_boot=1000, alpha=0.05):
    # Same descriptive bootstrap format as the old notebook; does not alter CCD outcomes.
    import numpy as np
    arr = np.asarray(flags, dtype=np.float32)
    if len(arr) == 0:
        return 0.0, 0.0, 0.0
    rng = np.random.default_rng(42)
    boots = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n_boot)]
    return float(arr.mean()), float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


def export_old_layout(run_dir, out_root, config, model_key, dataset_name, started):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    if out_root.exists():
        raise FileExistsError(f'Refusing to overwrite {out_root}')
    runs_dir, fig_dir = out_root / 'runs', out_root / 'figures'
    tab_dir, adp_dir = out_root / 'tables', out_root / 'adapters'
    for path in (runs_dir, fig_dir, tab_dir, adp_dir):
        path.mkdir(parents=True, exist_ok=False)
    stage_metrics = []
    for stage in ('pre_sft', 'post_sft'):
        metrics, predictions = [], []
        for condition in EXPECTED_CONFIGS:
            records = read_jsonl(run_dir / 'records' / f'{stage}__{condition}.jsonl')
            method = 'fewshot' if condition.startswith('fs') else ('sc' if condition.startswith('sc') else condition)
            fewshot_k = int(condition[2:]) if method == 'fewshot' else np.nan
            sc_k = float(condition[2:]) if method == 'sc' else np.nan
            strict_flags, relaxed_flags, format_flags, token_counts, disagree_flags = [], [], [], [], []
            for row in records:
                # Use the authoritative CCD winner-supported compliance/correctness fields.
                strict = int(row['strict_success'])
                relaxed = int(row['relaxed_correctness'])
                fmt = int(row['format_ok'])
                tokens = sum(int(candidate['generated_tokens']) for candidate in row['candidates'])
                disagree = int(len({candidate['vote_key'] for candidate in row['candidates']}) > 1) if method == 'sc' else np.nan
                pred = row.get('selected_code') if dataset_name == 'mbpp' else row.get('selected_prediction')
                predictions.append({'dataset': dataset_name, 'method': method, 'fewshot_k': fewshot_k,
                    'sc_k': sc_k, 'pred': pred, 'gold': row['gold'], 'strict_correct': strict,
                    'relaxed_correct': relaxed, 'format_ok': fmt, 'new_tokens': tokens, 'sc_disagree': disagree})
                strict_flags.append(strict); relaxed_flags.append(relaxed); format_flags.append(fmt); token_counts.append(tokens)
                if method == 'sc':
                    disagree_flags.append(disagree)
            strict_acc, strict_lo, strict_hi = bootstrap_ci(strict_flags)
            relaxed_acc, rel_lo, rel_hi = bootstrap_ci(relaxed_flags)
            metrics.append({'dataset': dataset_name, 'method': method, 'fewshot_k': fewshot_k, 'sc_k': sc_k,
                'n': len(records), 'strict_accuracy': strict_acc, 'strict_ci_low': strict_lo, 'strict_ci_high': strict_hi,
                'relaxed_accuracy': relaxed_acc, 'relaxed_ci_low': rel_lo, 'relaxed_ci_high': rel_hi,
                'format_success_rate': float(np.mean(format_flags)), 'avg_new_tokens': float(np.mean(token_counts)),
                'sc_disagreement_rate': float(np.mean(disagree_flags)) if disagree_flags else np.nan,
                'stage': stage, 'model_id': config['models'][model_key]})
        metrics_df, preds_df = pd.DataFrame(metrics), pd.DataFrame(predictions)
        metrics_path, preds_path = runs_dir / f'metrics_{stage}.csv', runs_dir / f'preds_{stage}.csv'
        metrics_df.to_csv(metrics_path, index=False)
        preds_df.to_csv(preds_path, index=False)
        stage_metrics.append(metrics_df)
        print('Saved:', metrics_path)
        print('Saved:', preds_path)
    main_metrics = pd.concat(stage_metrics, ignore_index=True)
    tag = f'{model_key}_{dataset_name}_uniform_ccd_v1'
    main_path = tab_dir / f'metrics_main_{tag}.csv'
    summary_path = tab_dir / f'summary_main_{tag}.csv'
    main_metrics.to_csv(main_path, index=False)
    main_metrics[['dataset', 'method', 'fewshot_k', 'sc_k', 'stage', 'strict_accuracy',
                  'relaxed_accuracy', 'format_success_rate', 'avg_new_tokens', 'sc_disagreement_rate']].to_csv(summary_path, index=False)
    try:
        plt.figure(figsize=(14, 6))
        plot_df = main_metrics.copy()
        plot_df['label'] = plot_df.apply(lambda r: f"{r['dataset']}\n{r['method']}"
            + (f" K={int(r['fewshot_k'])}" if pd.notna(r['fewshot_k']) else '')
            + (f" SC={int(r['sc_k'])}" if pd.notna(r['sc_k']) else ''), axis=1)
        order = list(plot_df[plot_df['stage'] == 'pre_sft']['label'].values)
        xpos = np.arange(len(order))
        pre_vals = [float(plot_df[(plot_df['stage'] == 'pre_sft') & (plot_df['label'] == label)]['strict_accuracy'].iloc[0]) for label in order]
        post_vals = [float(plot_df[(plot_df['stage'] == 'post_sft') & (plot_df['label'] == label)]['strict_accuracy'].iloc[0]) for label in order]
        width = 0.38
        plt.bar(xpos - width / 2, pre_vals, width=width, label='Pre-SFT')
        plt.bar(xpos + width / 2, post_vals, width=width, label='Post-SFT')
        plt.xticks(xpos, order, rotation=90)
        plt.ylabel('Strict Accuracy')
        plt.title(f'{model_key} on {dataset_name}: Pre vs Post SFT (Uniform CCD v1)')
        plt.legend()
        plt.tight_layout()
        fig_path = fig_dir / f'strict_accuracy_pre_vs_post_{tag}.png'
        plt.savefig(fig_path, dpi=220)
        plt.close()
        print('Saved figure to:', fig_path)
    except Exception as error:
        print('[warn] Could not make summary figure:', repr(error))
    shutil.copytree(run_dir / 'adapter', adp_dir / 'qlora_adapter')
    shutil.copytree(run_dir / 'records', runs_dir / 'raw_records')
    shutil.copytree(run_dir / 'reproducibility', out_root / 'reproducibility')
    for name in ('environment.json', 'run_metadata.json', 'stage_summaries.json', 'training_audit.json', 'run_complete.json'):
        shutil.copy2(run_dir / name, out_root / name)
    atomic_json(out_root / 'authoritative_config.json', config)
    pd.set_option('display.width', 220)
    pd.set_option('display.max_columns', 100)
    print(f'\n=== MAIN SUMMARY ({model_key} | {dataset_name} | UNIFORM CCD v1) ===')
    print(main_metrics.sort_values(['dataset','method','fewshot_k','sc_k','stage'],na_position='last').to_string(index=False))
    print('\nSaved thorough metrics to:', main_path)
    print('Saved summary table to:', summary_path)
    print('\nAll outputs saved under:', out_root)
    print(' - Runs:', runs_dir)
    print(' - Tables:', tab_dir)
    print(' - Figures:', fig_dir)
    print(' - Adapters:', adp_dir)
    print(f'\nTotal wall-clock time: {(time.time()-started)/60.0:.2f} minutes')


# -----------------------------
# 12) Online input preparation and reproducibility checks
# -----------------------------
def verify_frozen_manifest(manifest, dataset, config):
    value = dict(manifest)
    digest = value.pop('manifest_payload_sha256')
    if sha256_text(canonical_json(value)) != digest:
        raise ValueError('Frozen manifest content was modified.')
    if manifest['config_sha256'] != sha256_text(SOURCE_CONFIG_TEXT):
        raise ValueError('Frozen manifest belongs to a different experiment config.')
    if manifest['dataset'] != dataset or manifest['source'] != config['datasets'][dataset]:
        raise ValueError('Frozen dataset source does not match the authoritative config.')
    all_ids = set()
    for key, n in [('training', 300), ('demonstrations', 5), ('evaluation', 100)]:
        rows = manifest[key]
        if len(rows) != n:
            raise ValueError(f'{key} must contain exactly {n} examples.')
        for row in rows:
            item = dict(row)
            expected = item.pop('content_hash')
            if sha256_text(canonical_json(item)) != expected or row['item_id'] in all_ids:
                raise ValueError('Frozen item integrity failed or train/demo/evaluation IDs overlap.')
            all_ids.add(row['item_id'])


def resolve_model_revision(model_id, saved):
    if model_id in saved:
        entry = saved[model_id]
        if not re.fullmatch(r'[0-9a-f]{40}', entry['revision']):
            raise ValueError('Saved model revision is not an immutable commit.')
        return entry
    # Preserve the original Unsloth 4-bit repository mapping, then freeze that resolution.
    import inspect
    from unsloth.models.loader import get_model_name
    from huggingface_hub import HfApi
    options = {'load_in_4bit': True, 'load_in_fp8': False, 'token': None, 'trust_remote_code': False}
    accepted = inspect.signature(get_model_name).parameters
    resolved = get_model_name(model_id, **{k: v for k, v in options.items() if k in accepted})
    if not isinstance(resolved, str) or not resolved:
        raise RuntimeError(f'Unable to resolve original model: {model_id}')
    commit = HfApi().model_info(resolved, revision='main').sha
    if not commit or not re.fullmatch(r'[0-9a-f]{40}', commit):
        raise RuntimeError('Unable to freeze model commit.')
    return {'repo_id': resolved, 'revision': commit}


def validate_complete_block(run_dir, manifest, config):
    result = validate_run(run_dir, expected_n=100, raise_on_error=True)
    meta = load_json(run_dir / 'run_metadata.json')
    if meta['configurations'] != list(EXPECTED_CONFIGS):
        raise ValueError('All nine inference conditions are required.')
    ids = [r['item_id'] for r in manifest['evaluation']]
    hashes = [r['content_hash'] for r in manifest['evaluation']]
    for stage in ('pre_sft', 'post_sft'):
        for condition in EXPECTED_CONFIGS:
            rows = read_jsonl(run_dir / 'records' / f'{stage}__{condition}.jsonl')
            if [r['item_id'] for r in rows] != ids or [r['item_content_hash'] for r in rows] != hashes:
                raise ValueError('Evaluation differs from the frozen examples.')
            if any(sha256_text(r['prompt']) != r['prompt_sha256'] for r in rows):
                raise ValueError('Prompt integrity check failed.')
    audit = load_json(run_dir / 'training_audit.json')
    if [r['item_id'] for r in audit['item_audit']] != [r['item_id'] for r in manifest['training']]:
        raise ValueError('Training audit differs from the frozen examples.')
    return result


def write_replay_notebook(source, state, model_key, dataset_name, destination):
    """Embed data/versions as ordinary Python data; keep all code readable."""
    import pprint
    tree = ast.parse(source)
    replacements = {'REPLAY_STATE': pprint.pformat(state, width=110, sort_dicts=True),
                    'MODEL_KEY': repr(model_key), 'DATASET_NAME': repr(dataset_name),
                    'ACTION': repr('run')}
    lines = source.splitlines(keepends=True)
    edits = []
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name):
            name = node.targets[0].id
            if name in replacements:
                edits.append((node.lineno - 1, node.end_lineno, f'{name} = {replacements[name]}\n'))
    if len(edits) != len(replacements):
        raise RuntimeError('Cannot identify replay controls in this notebook source.')
    for start, end, replacement in sorted(edits, reverse=True):
        lines[start:end] = [replacement]
    frozen_source = ''.join(lines)
    compile(frozen_source, '<frozen notebook>', 'exec')
    notebook = {'nbformat': 4, 'nbformat_minor': 5, 'metadata': {
        'kernelspec': {'display_name': 'Python 3', 'language': 'python', 'name': 'python3'},
        'language_info': {'name': 'python'}}, 'cells': [{'cell_type': 'code',
        'id': 'ccd-readable-frozen', 'execution_count': None, 'metadata': {}, 'outputs': [],
        'source': frozen_source.splitlines(keepends=True)}]}
    atomic_json(destination, notebook)


def archive_zip(folder, destination):
    if destination.exists():
        raise FileExistsError(f'Refusing to overwrite {destination}')
    return shutil.make_archive(str(destination.with_suffix('')), 'zip',
                               root_dir=str(folder.parent), base_dir=folder.name)


def aggregate_uniform_runs(results_root, output_dir):
    config = json.loads(SOURCE_CONFIG_TEXT)
    expected = {f'{m}__{d}__seed42' for m in config['models'] for d in config['datasets']}
    actual = {p.name for p in results_root.iterdir() if p.is_dir()}
    if actual != expected:
        raise ValueError(f'Final aggregation needs the 60 raw blocks. Missing={sorted(expected-actual)}; extra={sorted(actual-expected)}')
    data_hashes, model_versions, environments = {}, {}, set()
    for name in sorted(expected):
        run_dir = results_root / name
        meta = load_json(run_dir / 'run_metadata.json')
        if meta['config_sha256'] != sha256_text(SOURCE_CONFIG_TEXT):
            raise ValueError('Cannot mix different scientific configurations.')
        if name != f'{meta["model_key"]}__{meta["dataset"]}__seed{meta["training_seed"]}':
            raise ValueError('Run folder and block identity differ.')
        manifest = load_json(run_dir / 'reproducibility' / 'manifest.json')
        verify_frozen_manifest(manifest, meta['dataset'], config)
        validate_complete_block(run_dir, manifest, config)
        data_hashes.setdefault(meta['dataset'], set()).add(manifest['manifest_payload_sha256'])
        replay = load_json(run_dir / 'reproducibility' / 'replay_state.json')
        model_versions.setdefault(meta['model_key'], set()).add(canonical_json(replay['model_revisions'][meta['model_id']]))
        environments.add(canonical_json({key: replay[key] for key in ('packages', 'python_major_minor', 'hardware')}))
    if (any(len(values) != 1 for values in data_hashes.values()) or len(environments) != 1
            or any(len(values) != 1 for values in model_versions.values())):
        raise ValueError('Runs used different samples, model revisions or runtime environments. Use the same frozen notebook across models.')
    aggregate_results(results_root, output_dir, expected_n=100)


# -----------------------------
# 13) Execute one complete model × dataset × seed block
# -----------------------------
def run_notebook():
    global MODEL_REVISIONS
    started = time.time()
    work, output = Path(WORK_ROOT), Path(OUTPUT_ROOT)
    work.mkdir(parents=True, exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    config = json.loads(SOURCE_CONFIG_TEXT)
    validate_config(config)
    if sha256_text(SOURCE_CONFIG_TEXT) != SOURCE_CONFIG_SHA256:
        raise ValueError('The source experiment configuration has been edited.')
    if ACTION not in ('run', 'aggregate'):
        raise ValueError('ACTION must be run or aggregate.')
    if ACTION == 'aggregate':
        destination = work / 'aggregate_v1'
        aggregate_uniform_runs(Path(AGGREGATE_RESULTS_ROOT), destination)
        print('Final results:', archive_zip(destination, output / 'aggregate_v1.zip'))
        return
    if MODEL_KEY not in config['models'] or DATASET_NAME not in config['datasets']:
        raise ValueError('MODEL_KEY and DATASET_NAME must match the source config.')
    if DATASET_NAME == 'mbpp' and not ALLOW_CODE_EXECUTION:
        raise ValueError('For MBPP, set ALLOW_CODE_EXECUTION=True explicitly in an isolated Kaggle runtime.')
    job_id = f'{MODEL_KEY}__{DATASET_NAME}__seed42'
    export_name = f'llm_reasoning_outputs_{MODEL_KEY}_{DATASET_NAME}_uniform_ccd_v1'
    run_dir, export_dir = work / 'results' / job_id, output / export_name
    for path in (run_dir, export_dir, output / f'{job_id}.zip', output / f'{export_name}.zip'):
        if path.exists():
            raise FileExistsError(f'Existing result: {path}. Preserve it and use a fresh output location for a new attempt.')
    source = notebook_source()
    state = json.loads(json.dumps(REPLAY_STATE)) if REPLAY_STATE else {
        'manifests': {}, 'dataset_revisions': {}, 'model_revisions': {}}
    if REPLAY_STATE and state['config_sha256'] != SOURCE_CONFIG_SHA256:
        raise ValueError('Replay state belongs to a different scientific protocol.')
    ensure_dependencies(work, REPLAY_STATE)
    # Match the successful old notebook's import order; never import Trainer first.
    import unsloth
    import torch
    import numpy
    import pandas
    import matplotlib
    from datasets import load_dataset
    from transformers import Trainer, TrainingArguments
    signature = gpu_signature()
    packages = runtime_packages()
    if REPLAY_STATE:
        if packages != state['packages']:
            raise RuntimeError('Runtime package versions differ from the saved replay environment.')
        if signature != state['hardware']:
            raise RuntimeError('Replay requires the original accelerator configuration and Torch CUDA build.')
    state.update(config_sha256=SOURCE_CONFIG_SHA256, packages=packages,
                 python_major_minor=list(sys.version_info[:2]), hardware=signature)
    config_path = work / 'experiment_config.json'
    if config_path.exists() and config_path.read_bytes() != SOURCE_CONFIG_TEXT.encode('utf-8'):
        raise ValueError('Existing configuration differs from the original ZIP.')
    config_path.write_text(SOURCE_CONFIG_TEXT, encoding='utf-8')
    manifest_dir = work / 'manifests'
    manifest_dir.mkdir(exist_ok=True)
    manifest_path = manifest_dir / f'{DATASET_NAME}.json'
    if DATASET_NAME in state['manifests']:
        manifest = state['manifests'][DATASET_NAME]
        verify_frozen_manifest(manifest, DATASET_NAME, config)
        if manifest_path.exists() and load_json(manifest_path) != manifest:
            raise ValueError('Local manifest differs from embedded frozen data.')
        if not manifest_path.exists():
            atomic_json(manifest_path, manifest)
        print('[data] Using exact embedded frozen examples; no dataset re-download.')
    elif manifest_path.exists():
        manifest = load_json(manifest_path)
        verify_frozen_manifest(manifest, DATASET_NAME, config)
        revisions_path = work / f'{DATASET_NAME}_dataset_revision.json'
        if not revisions_path.exists():
            raise ValueError('Existing manifest has no saved source revision. Preserve it and use a fresh WORK_ROOT.')
        state['dataset_revisions'][DATASET_NAME] = load_json(revisions_path)
        state['manifests'][DATASET_NAME] = manifest
    else:
        from huggingface_hub import HfApi
        spec = config['datasets'][DATASET_NAME]
        source_ref = spec.get('revision') or (
            'refs/convert/parquet' if DATASET_NAME == 'logiqa' else 'main'
        )
        commit = HfApi().dataset_info(spec['path'], revision=source_ref).sha
        if not commit or not re.fullmatch(r'[0-9a-f]{40}', commit):
            raise RuntimeError('Unable to freeze the dataset revision.')
        state['dataset_revisions'][DATASET_NAME] = {'repo_id': spec['path'], 'revision': commit}
        print(f'[data] Downloading {spec["path"]} at {commit}')
        prepare_selected_manifest(config_path, manifest_dir, DATASET_NAME, commit)
        manifest = load_json(manifest_path)
        verify_frozen_manifest(manifest, DATASET_NAME, config)
        state['manifests'][DATASET_NAME] = manifest
        atomic_json(work / f'{DATASET_NAME}_dataset_revision.json', state['dataset_revisions'][DATASET_NAME])
    model_id = config['models'][MODEL_KEY]
    state['model_revisions'][model_id] = resolve_model_revision(model_id, state['model_revisions'])
    MODEL_REVISIONS = state['model_revisions']
    replay_path = output / f'{MODEL_KEY}_{DATASET_NAME}_FROZEN.ipynb'
    if replay_path.exists():
        raise FileExistsError(f'Refusing to overwrite saved replay: {replay_path}')
    # Save replay before training too: failed attempts retain the exact planned inputs.
    write_replay_notebook(source, state, MODEL_KEY, DATASET_NAME, replay_path)
    assignments = work / 'assignments.csv'
    if not assignments.exists():
        create_assignments(config_path, assignments)
    print(f'\nRunning {job_id}: 300 train / 5 demos / 100 paired eval / 3 epochs / 9 conditions')
    completed = run_experiment(config_path, manifest_dir, work / 'results', MODEL_KEY,
                               DATASET_NAME, 42, ALLOW_CODE_EXECUTION and DATASET_NAME == 'mbpp')
    validation = validate_complete_block(completed, manifest, config)
    audit = completed / 'reproducibility'
    audit.mkdir()
    shutil.copy2(config_path, audit / 'experiment_config.json')
    shutil.copy2(manifest_path, audit / 'manifest.json')
    shutil.copy2(replay_path, audit / replay_path.name)
    shutil.copy2(assignments, audit / 'assignments.csv')
    (audit / 'notebook_source.py').write_text(source, encoding='utf-8')
    (audit / 'README_source.md').write_text(SOURCE_README_TEXT, encoding='utf-8')
    (audit / 'requirements-lock.txt').write_text(''.join(f'{k}=={v}\n' for k,v in packages.items()), encoding='utf-8')
    atomic_json(audit / 'runtime_packages.json', packages)
    atomic_json(audit / 'replay_state.json', state)
    freeze_environment(audit / 'requirements-full-environment.txt')
    print('Raw scientific archive:', archive_zip(completed, output / f'{job_id}.zip'))
    export_old_layout(completed, export_dir, config, MODEL_KEY, DATASET_NAME, started)
    print('Presentation archive:', archive_zip(export_dir, output / f'{export_name}.zip'))
    print('Validated:', validation['valid'])
    print('Frozen replay notebook:', replay_path)



# Appendix: original protocol text included in the output audit archive.
SOURCE_README_TEXT = r'''# Uniform CCD Experiment Pipeline

This package runs a controlled, reproducible version of the Compliance–Correctness Decomposition (CCD) study across all declared models and datasets.

Every experimental block uses:

- one base model;
- one dataset;
- one training seed;
- one independently trained QLoRA adapter;
- exactly 300 frozen SFT examples;
- exactly 100 frozen evaluation items before SFT;
- the same 100 item IDs after SFT;
- the same nine inference configurations;
- the same training recipe and evaluators.

Do **not** combine multiple datasets into one adapter. Do **not** manually change settings for a particular model or dataset. A model–dataset–seed block is the unit of intervention; the nine inference configurations are repeated evaluation conditions, not independent training experiments.

## 1. Files

```text
ccd_uniform_pipeline/
├── ccd_uniform.ipynb       # Self-contained Kaggle/Jupyter experiment notebook
├── experiment_config.json # Authoritative models and hyperparameters
├── requirements.txt       # Bootstrap dependencies
└── README.md               # This protocol
```

The notebook is the authoritative executable. It contains separate, clearly
labeled cells for dependency installation, pipeline implementation,
coordinator setup, worker execution, validation, and aggregation. Run the
implementation cells before enabling any action cell.

The coordinator will additionally create:

```text
manifests/                  # Frozen records; distribute unchanged to every worker
assignments.csv             # Job allocation table
requirements-lock.txt       # Exact validated software environment
```

## 2. Scientific design

For 15 models, four datasets, nine inference configurations, and 100 paired evaluation items, one training seed produces:

```text
15 models × 4 datasets = 60 independent adapters
60 adapters × 9 configurations = 540 evaluation cells
540 cells × 100 items = 54,000 matched pre/post response pairs
```

The datasets are:

- GSM8K: numerical reasoning;
- StrategyQA: Boolean reasoning;
- LogiQA: multiple-choice logical reasoning;
- MBPP: Python program synthesis.

The configurations are fixed in this exact order:

```text
zero, cot, fs0, fs1, fs3, fs5, sc1, sc3, sc5
```

`FS0` uses the few-shot scaffold with an empty demonstration block. `SC1`, `SC3`, and `SC5` use sampled candidate generation and winner-supported compliance.

## 3. Uniform hyperparameters

| Component | Fixed value |
|---|---:|
| Training examples | 300 per dataset |
| Evaluation items | 100 before and 100 after SFT |
| Demonstration pool | 5 reserved training-partition examples |
| Training epochs | 3 |
| Learning rate | `1e-4` |
| Per-device batch size | 1 |
| Gradient accumulation | 16 |
| Effective batch size | 16 |
| Warmup ratio | 3% |
| LR schedule | Cosine |
| Optimizer | AdamW 8-bit |
| Weight decay | 0 |
| Maximum gradient norm | 1.0 |
| LoRA rank | 16 |
| LoRA alpha | 16 |
| LoRA dropout | 0.05 |
| Weight loading | 4-bit |
| Maximum sequence length | 2,048 tokens |
| Primary data seed | 42 |
| Primary training seed | 42 |

LoRA targets are:

```text
q_proj, k_proj, v_proj, o_proj,
gate_proj, up_proj, down_proj
```

Generation budgets are dataset-specific but identical across models:

| Dataset | Maximum new tokens |
|---|---:|
| StrategyQA | 40 |
| LogiQA | 96 |
| GSM8K | 160 |
| MBPP | 384 |

Self-consistency uses temperature `1.0`, top-p `0.95`, and top-k `20`.

## 4. Why epochs replace a fixed 300-step budget

Using 300 optimizer steps with only 300 training examples and an effective batch size of 16 corresponds to approximately 16 effective passes over the dataset. This can create unequal overfitting when dataset sizes differ. The uniform pipeline uses three epochs and records both assistant-token exposure and truncation statistics.

Do not change the pipeline back to a fixed step count without defining a new experiment version.

## 5. Coordinator setup

Only the coordinator prepares the manifests and validated environment. Workers must not independently select data.

### 5.1 Open the notebook in a fresh environment

Upload `ccd_uniform.ipynb` to Kaggle and attach the project files as a Kaggle
dataset. In the notebook's dependency cell, set:

```python
INSTALL_DEPENDENCIES = True
```

Run that cell once, restart the Kaggle kernel, and then set the flag back to
`False`. Unsloth must be imported before Transformers training classes; the
notebook handles this internally.

### 5.2 Freeze the datasets

In the notebook's shared-path cell, keep the uploaded configuration path and
send new manifests to writable Kaggle storage:

```python
CONFIG_PATH = Path("/kaggle/input/ccd-uniform-pipeline/experiment_config.json")
MANIFEST_DIR = Path("/kaggle/working/ccd_uniform/manifests")
```

In the coordinator control cell, set `PREPARE_MANIFESTS = True`, run the cell
once, and immediately return the flag to `False`.

This command:

1. downloads the official train and test splits;
2. normalizes every example;
3. selects 300 SFT examples with the declared data seed;
4. reserves five different examples for few-shot prompting;
5. selects exactly 100 held-out evaluation items;
6. asserts that training, demonstrations, and evaluation do not overlap;
7. stores full canonical records, source fingerprints, hashes, and IDs.

The code never falls back to a training split for evaluation. If a required test split is unavailable, it stops.

After creating the manifests, do not regenerate them. Create one read-only
Kaggle dataset containing the unchanged `experiment_config.json`, the complete
`manifests/` directory, and the validated environment lock. Every worker must
attach that exact dataset.

### 5.3 Run one smoke-test block

Use a small model but retain the complete scientific configuration. In the
worker control cell, set:

```python
RUN_JOB = True
MODEL_KEY = "gemma3_270m"
DATASET_NAME = "strategyqa"
TRAINING_SEED = 42
ALLOW_CODE_EXECUTION = False
```

After completion, set `RUN_JOB = False`. In the validation cell, set:

```python
RUN_VALIDATION = True
RUN_DIR = RESULTS_ROOT / "gemma3_270m__strategyqa__seed42"
```

The validation result must contain:

```json
{"valid": true}
```

### 5.4 Freeze the exact environment

Only after the smoke run succeeds, use the coordinator cell:

```python
FREEZE_ENVIRONMENT = True
```

Run it once and return the flag to `False`.

Distribute this lock file to every worker. Workers should install it in a fresh environment:

```bash
python -m pip install -r requirements-lock.txt
```

All workers must use the same Kaggle accelerator type when practical. Every run automatically records package versions, GPU type, CUDA visibility, Python version, and platform information in `environment.json`.

### 5.5 Create the job matrix

In the coordinator cell, set:

```python
CREATE_ASSIGNMENTS = True
```

Run it once and return the flag to `False`.

The default configuration creates 60 jobs. Fill only the `assigned_to` and `status` columns. Do not edit model keys, dataset names, or seeds.

## 6. Worker instructions

Each worker receives:

- this complete project directory;
- the coordinator’s frozen `manifests/` directory;
- `requirements-lock.txt`;
- one or more rows from `assignments.csv`.

### 6.1 Run an assigned job

Open `ccd_uniform.ipynb`, run all implementation cells, confirm the shared
paths, and edit only the declared worker parameters:

```python
RUN_JOB = True
MODEL_KEY = "qwen35_08b"
DATASET_NAME = "gsm8k"
TRAINING_SEED = 42
ALLOW_CODE_EXECUTION = False
```

Run the worker cell once. After the job finishes, set `RUN_JOB = False`.
Available model keys are listed in `experiment_config.json`.

### 6.2 MBPP command and safety warning

MBPP executes generated Python. Run it only in a disposable, isolated Kaggle runtime. The pipeline uses a subprocess, timeout, isolated Python mode, temporary working directory, and operating-system resource limits, but these measures are **not a complete security sandbox**.

MBPP requires explicit acknowledgment in the worker cell:

```python
RUN_JOB = True
MODEL_KEY = "qwen35_08b"
DATASET_NAME = "mbpp"
TRAINING_SEED = 42
ALLOW_CODE_EXECUTION = True
```

Do not execute generated programs on a personal computer, shared server, or machine containing credentials or valuable files.

### 6.3 Validate every job

```python
RUN_VALIDATION = True
RUN_DIR = RESULTS_ROOT / "MODEL_KEY__DATASET__seed42"
```

Never submit a result folder that does not report `"valid": true`.

### 6.4 What to return

Compress the entire completed run folder, including:

```text
adapter/
records/
environment.json
run_metadata.json
stage_summaries.json
training_audit.json
run_complete.json
```

Do not return only screenshots or summary CSV files. Raw prompts, candidates, completions, parser decisions, and execution outcomes are required for the paper audit.

## 7. Output structure

One completed job has this structure:

```text
results/
└── qwen35_08b__gsm8k__seed42/
    ├── adapter/
    ├── records/
    │   ├── pre_sft__zero.jsonl
    │   ├── post_sft__zero.jsonl
    │   ├── ...
    │   ├── pre_sft__sc5.jsonl
    │   └── post_sft__sc5.jsonl
    ├── environment.json
    ├── run_metadata.json
    ├── stage_summaries.json
    ├── training_audit.json
    └── run_complete.json
```

Each item-level record stores:

- stable item ID and content hash;
- model, dataset, adapter, seed, stage, and configuration;
- raw prompt and prompt hash;
- original and retained prompt-token counts;
- prompt truncation status;
- all raw self-consistency candidates;
- candidate-specific sampling seed, parsed answer, code, compliance, token count, and ceiling status;
- selected winner and selected candidate;
- winner-supported compliance;
- historical any-sample compliance for sensitivity analysis;
- relaxed correctness;
- strict success;
- MBPP execution status and error class.

## 8. Evaluation definitions

For every selected output:

```text
F = 1 if a contract-compliant candidate supports the selected winner
C = 1 if the selected answer is correct under the task criterion
S = F × C
V = F − S
G = ΔF − ΔS = ΔV
```

For MBPP, correctness is execution against the frozen benchmark assertions. Empty code, syntax errors, runtime failures, timeouts, and failed assertions remain in the denominator.

The StrategyQA relaxed parser never defaults to `No`. Ambiguous outputs containing both Yes and No are unparseable. LogiQA outputs containing multiple distinct option letters are also unparseable.

For self-consistency, the pipeline:

1. parses each candidate independently;
2. finds modal vote keys;
3. resolves a tie using the earliest candidate among the tied modal answers;
4. never selects a nonmodal answer;
5. prefers a compliant candidate among supporters of the winning answer;
6. defines compliance using the winner’s supporters, not an unrelated sample.

## 9. Aggregating returned jobs

Place every validated run folder directly under a single `results/` directory:

```text
results/
├── gemma3_270m__gsm8k__seed42/
├── gemma3_270m__strategyqa__seed42/
└── ...
```

In the notebook's aggregation cell, set:

```python
RUN_AGGREGATION = True
AGGREGATE_RESULTS_ROOT = Path("/kaggle/working/results")
AGGREGATE_OUTPUT_DIR = Path("/kaggle/working/aggregate_v1")
```

Run the cell once and return `RUN_AGGREGATION` to `False`.

The command first validates every run and stops if any run is invalid. It then produces:

```text
aggregate_v1/
├── validation.csv
├── cell_summary.csv
├── block_summary.csv
└── overall_summary.json
```

`cell_summary.csv` has one row per model–dataset–seed–configuration cell. `block_summary.csv` averages the nine inference configurations within each independently trained adapter. The primary overall estimate gives equal weight to adapter blocks, not to individual generations.

`overall_summary.json` also reports model-cluster bootstrap intervals using 10,000 replicates.

## 10. Collaboration rules

Every contributor must follow these rules:

1. Use the distributed `experiment_config.json` without editing it.
2. Use the distributed frozen manifests without regenerating them.
3. Use the distributed exact environment lock.
4. Run only assigned model–dataset–seed combinations.
5. Do not change batch size unless the coordinator declares a new experiment version. Batch changes can alter sampled generation.
6. Do not change prompts, chat templates, generation budgets, parsers, tests, or stopping rules.
7. Do not manually repair individual model outputs.
8. Do not rerun only unfavorable evaluation configurations.
9. Keep failed or timed-out MBPP outputs in the denominator.
10. Return complete validated run folders.

If a job fails because of an out-of-memory error or model-specific incompatibility, send the complete error log to the coordinator. Do not silently change the configuration.

## 11. Reproducibility checks enforced by the code

The validator checks that:

- every cell contains exactly 100 pre-SFT and 100 post-SFT records;
- the pre- and post-SFT item IDs match exactly and in the same order;
- prompt hashes match before and after SFT;
- item IDs are unique;
- raw prompts and candidate completions are present;
- strict success never exceeds compliance;
- all nine configurations are present.

The runner refuses to overwrite an existing run directory. This prevents accidental replacement of valid results.

## 12. Seed replication

The default main experiment uses training seed 42 for all 60 blocks. For a stronger seed-robustness study, add seeds to the authoritative configuration only before preparing assignments, for example:

```json
"training_seeds": [42, 123, 2026]
```

Using three seeds for all blocks creates 168 training runs. A practical alternative is to create a separate experiment version that repeats three seeds for four representative models across all datasets. Do not mix undeclared extra seeds into the main result directory.

## 13. Optional format-control experiment

The strongest follow-up experiment compares two otherwise identical SFT conditions:

1. contract-matched targets containing `Final Answer:`;
2. format-neutral targets containing the same answers without the evaluator’s marker.

This ablation requires a new configuration and separate output directory because it changes the intervention. It should not be mixed with `ccd_uniform_v1` results.

## 14. Human annotation

Human annotation is not required for the primary quantities `F`, `S`, `V`, and `G`. It is optional for validating relaxed extraction from noncompliant natural-language responses. MBPP uses execution-based correctness.

## 15. Reporting the experiment in the paper

Describe the controlled experiment as follows:

> We trained one independent QLoRA adapter for each model–dataset–seed block using a common recipe. Every block used 300 frozen training examples, three epochs, an effective batch size of 16, learning rate (10^{-4}), LoRA rank and alpha 16, and the same nine inference configurations. Evaluation used exactly 100 fixed held-out items before and after SFT. Pre- and post-SFT prompts, demonstrations, decoding settings, token budgets, item IDs, contract checkers, and correctness procedures were identical within each paired comparison. Inference configurations were treated as repeated evaluations of an adapter rather than independent training interventions.

Do not claim that 540 evaluation cells are 540 independent SFT experiments. With one seed, the study contains 60 independently trained adapter blocks.

## 16. Troubleshooting

### Existing run directory

The pipeline never overwrites results. If a previous attempt failed, preserve
it for diagnosis and use a new notebook path such as:

```python
RESULTS_ROOT = Path("/kaggle/working/results_retry1")
```

### Missing dataset split

Stop and inform the coordinator. Do not substitute the training split.

### Out of memory

Do not change the batch size independently. Report the model, dataset, GPU, configuration, and error. The coordinator must decide whether to create a new experiment version with a universally smaller batch size.

### Model-specific chat-template error

Do not hand-edit prompts. Report the full traceback and recorded package versions.

### MBPP execution warning

Use a fresh disposable Kaggle session and set
`ALLOW_CODE_EXECUTION = True` only for the assigned MBPP job. Never run
generated code in a trusted environment.
'''

run_notebook()
